## Notebook Guide (Preprocessing.ipynb)

This notebook contains intial EDA on unprocessed data and Main Preprocessing steps(data_pipeline) that prepare cleaned data for `model.ipynb`

Sections
- Data loading and session setup
- Feature engineering helpers and data QA
- EDA: temporal, spatial, weather, and outlier analysis
- Modeling hooks (optional): reuse pipelines and evaluate
- Reporting/exports

Run order
1) Session + data load
2) Feature engineering helpers (only those you need)
3) EDA modules (temporal → spatial → weather → outliers), this also preprocess the external datasets
4) Main Preprocessing step: data_pipeline!!!
5) Reports/exports

Notes
- Each EDA block is modular; you can run them independently
- Prefer sampling for heavy plots; avoid full `toPandas()` on large frames


### Data prep & assumptions

- All feature derivations mirror the logic in `model.ipynb` unless noted
- Avoid target leakage when adding derived metrics
- Use stratified or time-based sampling for heavy analyses
- Keep memory in mind on WSL; prefer sampled plots and `cache()` only when reused

Checklist before EDA
- Spark session up and stable (reasonable `driver.memory`, partitions)
- Input columns present (IDs, zones, times, weather)
- Optional: join lookup for zone metadata if you need grouped views (Airports/Boroughs)


### Modeling & outputs notes (optional hooks)

- Reuse trained pipelines from `model.ipynb` (load from `lake/models/...`)
- Keep train/test splits and preprocessing aligned with the main notebook
- Store outputs under `lake/reports/` or `lake/models/` with versioned names

Good practices
- Log key params and metrics next to artifacts
- Keep heavy training out of this notebook; use it for light evaluation/visualization
- Document assumptions alongside each output (one line per figure/table)


In [9]:
# High-memory SparkSession setup for 64 GB RAM machines
# Run this cell once at the start of the notebook
from pyspark.sql import SparkSession

# Stop an existing session if present to apply new configs
try:
    spark.stop()
except Exception:
    pass

spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 2 - HighMem")
    .master("local[*]")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    # Memory and results
    .config("spark.driver.memory", "40g")            # leave ~16g for OS/other apps
    .config("spark.driver.maxResultSize", "8g")
    # Performance tuning
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.shuffle.partitions", "400")   # increase parallelism on local[*]
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "1024m")
    .config("spark.memory.fraction", "0.6")
    .getOrCreate()
)

import pyspark
print("Spark version:", pyspark.__version__)
print("Spark UI:", spark.sparkContext.uiWebUrl)



Spark version: 4.0.0
Spark UI: http://10.255.255.254:4041


25/08/31 21:37:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [10]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T
from pyspark.sql.functions import broadcast
from datetime import datetime
from pyspark import StorageLevel

In [11]:
# Stop any existing Spark session to avoid stale/failed JVM gateway issues
try:
    spark.stop()
except Exception:
    pass


In [12]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 2")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config('spark.driver.memory', '3g')
    .config('spark.executor.memory', '2g')
    .getOrCreate()
)


25/08/31 21:37:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Landing Preprocessing


In [28]:
LANDING_HVFHS = "/lake/landing/hvfhs"                      # year=YYYY/month=MM/*.parquet
RAW_HVFHS     = "/lake/raw/hvfhs/v1"
CUR_HVFHS     = "/lake/curated/hvfhs_core/v1"
METRICS_PATH  = "/lake/curated/metrics/hvfhs_quality/v1"
DEV_HOURLY    = "/lake/dev/facts/hvfhs_hourly_pu/v1"

LANDING_TLC_ZONES = "/lake/landing/tlc_zone_lookup/version=2024-01-01"
LANDING_MTA_CBD   = "/lake/landing/mta_cbd_mapping/version=2024-01-01"

DATA_VERSION     = "2025-08-13"
PIPELINE_VERSION = "v1"


In [ ]:
sdf_2023 = spark.read.parquet("data/2023/")
sdf_2024 = spark.read.parquet("data/2024/")

sdf_2023.printSchema()
sdf_2024.printSchema()

sdf_2023.show(5)
sdf_2024.show(5)

# Raw  processing


In [ ]:
DATA_BASE   = "./data"                     # Root directory of your existing data (contains 2023/ and 2024/)
RAW_HVFHS   = "./lake/raw/hvfhs/v1"        # Raw layer output directory (use a clean location)

In [ ]:
def yn_to_bool(col):
    """
    Converts a column containing 'Y'/'N' (case-insensitive) values to boolean True/False.

    Args:
        col (str): The name of the column to convert.

    Returns:
        Column: A Spark Column expression that is True for 'Y', False for 'N', and None otherwise.
    """
    return F.when(F.upper(F.col(col)) == F.lit("Y"), F.lit(True)) \
            .when(F.upper(F.col(col)) == F.lit("N"), F.lit(False)) \
            .otherwise(F.lit(None).cast("boolean"))

def write_part(df, path, mode="append", parts=("year","month")):
    """
    Writes a DataFrame to the specified path in Parquet format, partitioned by the given columns.

    Args:
        df (DataFrame): The Spark DataFrame to write.
        path (str): The destination path for the Parquet files.
        mode (str, optional): Write mode ('append', 'overwrite', etc.). Defaults to "append".
        parts (tuple, optional): Columns to partition by. Defaults to ("year", "month").

    Minimizes shuffle by coalescing to a reasonable number of output files
    after partitioning columns are set.
    """
    parts_cols = list(parts)
    df_to_write = df
    if parts_cols:
        df_to_write = df.repartition(*parts_cols)
    df_to_write = df_to_write.coalesce(64)
    (df_to_write.write
        .mode(mode)
        .partitionBy(*parts_cols)
        .parquet(path))

def load_dims():
    def load_dims():
        """
        Loads and processes dimension tables for TLC zones and MTA CBD mapping.

        Returns:
            tlc (DataFrame): TLC zone lookup table with columns:
                - LocationID (int)
                - Borough (str)
                - Zone (str)
                - service_zone (str)
            cbd (DataFrame): MTA CBD mapping table with columns:
                - LocationID (int)
                - in_cbd (int, always 1)
        """
        tlc = (spark.read.csv(LANDING_TLC_ZONES, header=True)
               .select(F.col("LocationID").cast("int").alias("LocationID"),
                       "Borough","Zone","service_zone")
               .dropna(subset=["LocationID"])
               .dropDuplicates(["LocationID"]))
        cbd = (spark.read.csv(LANDING_MTA_CBD, header=True)
               .select(F.col("LocationID").cast("int").alias("LocationID"))
               .dropDuplicates(["LocationID"])
               .withColumn("in_cbd", F.lit(1)))
        return tlc, cbd

In [ ]:
def read_month_df(year: int, month: int):
    y  = f"{year}"
    mm = f"{month:02d}"
    patterns = [
        # Primary naming pattern (one file per month)
        f"{DATA_BASE}/{y}/fhvhv_tripdata_{y}-{mm}.parquet",
        # Fallback: if a month is split into multiple files (rare), use wildcard
        f"{DATA_BASE}/{y}/fhvhv_tripdata_{y}-{mm}-*.parquet",
        # Broader fallback: any parquet in the folder that includes yyyy-mm
        f"{DATA_BASE}/{y}/*{y}-{mm}*.parquet",
        # Compatibility for month subfolders or alternate naming
        f"{DATA_BASE}/{y}/{mm}/*.parquet",
        f"{DATA_BASE}/{y}/{y}-{mm}.parquet",
    ]
    last_err = None
    for p in patterns:
        try:
            return spark.read.parquet(p)
        except Exception as e:
            last_err = e
            continue
    raise last_err

In [ ]:
def build_raw_from_landing(year:int, month:int):
    df = read_month_df(year, month)
    df = (df
          .withColumn("PULocationID", F.col("PULocationID").cast("int"))
          .withColumn("DOLocationID", F.col("DOLocationID").cast("int"))
          .withColumn("trip_miles",   F.col("trip_miles").cast("double"))
          .withColumn("trip_time",    F.col("trip_time").cast("int"))
          .withColumn("base_passenger_fare", F.col("base_passenger_fare").cast("double"))
          .withColumn("tolls",               F.col("tolls").cast("double"))
          .withColumn("bcf",                 F.col("bcf").cast("double"))
          .withColumn("sales_tax",           F.col("sales_tax").cast("double"))
          .withColumn("congestion_surcharge",F.col("congestion_surcharge").cast("double"))
          .withColumn("airport_fee",         F.col("airport_fee").cast("double"))
          .withColumn("tips",                F.col("tips").cast("double"))
          .withColumn("driver_pay",          F.col("driver_pay").cast("double"))
          .withColumn("year",  F.year("pickup_datetime"))
          .withColumn("month", F.month("pickup_datetime"))
          .withColumn("ingest_ts", F.current_timestamp())
    )
    df = df.persist(StorageLevel.DISK_ONLY)
    write_part(df, RAW_HVFHS, mode="overwrite")
    count_rows = df.count()
    df.unpersist()
    return count_rows


In [ ]:
# ==== 4) Run 2024-06 and perform quick sanity checks ====
rows = build_raw_from_landing(2024, 6)
print(f"[RAW] 2024-06 rows written: {rows:,}")

raw_2406 = (spark.read.parquet(RAW_HVFHS)
            .where("year = 2024 AND month = 6"))

print("Sample 10 rows:")
(raw_2406
 .select("pickup_datetime","dropoff_datetime","PULocationID","DOLocationID",
         "trip_miles","base_passenger_fare")
 .show(10, truncate=False))

print("Time range and row count checks:")
raw_2406.select(
    F.min("pickup_datetime").alias("min_pickup"),
    F.max("pickup_datetime").alias("max_pickup"),
    F.count("*").alias("rows")
).show(truncate=False)


In [ ]:
def run_all_raw():
    # Use dynamic partition overwrite (overwrite only current partition)
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

    for y in [2023, 2024]:
        for m in range(1, 13):
            try:
                print(f"→ RAW {y}-{m:02d} ...", end=" ")
                build_raw_from_landing(y, m)
                print("OK")
            except Exception:
                print("SKIP")
                traceback.print_exc()

    # Lightweight validation: list successfully written partitions
    (spark.read.parquet(RAW_HVFHS)
        .select("year", "month")
        .distinct()
        .orderBy("year", "month")
        .show(48, truncate=False))
          

In [ ]:
run_all_raw()

# Curated layer preprocessing


In [ ]:
# ------- Config -------
RAW_HVFHS          = "./lake/raw/hvfhs/v1"
CUR_HVFHS_UBER     = "./lake/curated/hvfhs_core_uber/v1"
METRICS_PATH_UBER  = "./lake/curated/metrics/hvfhs_quality_uber/v1"

TLC_ZONES_CSV      = "./lookup/taxi_zone_lookup.csv"     # TLC lookup (LocationID,Borough,Zone,service_zone)
TLC_ZONES_GEOJSON  = "./lookup/taxi_zones.geojson"       # TLC polygons (has LocationID)
MTA_CBD_GEOJSON    = "./lookup/mta_cbd.geojson"          # CBD polygon(s)
MTA_CBD_CSV        = "./lookup/mta_cbd_locationids.csv"  # will be generated if missing

PROVIDER_CODE      = "HV0003"     # Uber
SPEED_LIMIT_MAX_AVG_MPH = 50.0    # hard cap for avg trip speed
CBD_OVERLAP_RATIO_THRESH = 0.01   # 1% area overlap threshold for marking in_cbd
CBD_PLANAR_CRS     = 2263         # EPSG:2263 (NY Long Island), good for area ops


In [20]:
import os
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

### EDA modules overview: 
NOTE: EDA here is done based on not processed data, where EDA on feature.ipynb files run EDA on data after preprocessing

Temporal
- Pickup hour/day/month/season/period; weekend/holiday; rush hour
- Use sampled counts and normalized rates; annotate with significant events if relevant

Spatial
- PU/DO zones, boroughs, and cross-borough flows
- Optional: aggregate to Airports/Manhattan/boroughs using the lookup

Weather
- Temperature bands (freezing/hot), precipitation, snow, wind, comfort index
- Overlay with temporal patterns to probe interactions

Outliers & data quality
- Distribution checks for fares, durations, speeds
- Rule-based filters and explainable flags

Tips
- Use `.sample(fraction)` before `toPandas()`
- Use cached intermediates only when referenced multiple times


### Explortory analysis + outlier detection

In [ ]:
def extract_uber_data_for_analysis(year_list=None, month_list=None, sample_fraction=None):
    """
    Simplified version using existing trip_time field 
    """
    print("=== EXTRACTING UBER DATA (SIMPLIFIED VERSION) ===")
    
    if year_list is None:
        year_list = [2023, 2024]
    if month_list is None:
        month_list = list(range(1, 13))
    
    print(f"Loading data for years: {year_list}, months: {month_list}")
    
    # Load and filter data
    time_filter = F.col("year").isin(year_list) & F.col("month").isin(month_list)
    raw_all = spark.read.parquet(RAW_HVFHS).filter(time_filter)
    
    # Filter for Uber only
    lic = F.upper(F.trim(F.col("hvfhs_license_num").cast("string")))
    is_uber = (lic == F.lit(PROVIDER_CODE))
    uber_raw = raw_all.filter(is_uber)
    
    print(f"Uber trips found: {uber_raw.count():,}")
    
    # Use existing trip_time instead of calculating duration_sec
    uber_analysis = (uber_raw
        .withColumn("duration_sec", F.col("trip_time"))  # Use existing field!
        .withColumn("duration_hours", F.col("trip_time") / 3600.0)
        .withColumn("mph", 
                   F.when(F.col("trip_time") > 0, 
                         F.col("trip_miles") / (F.col("trip_time") / 3600.0)))
        # Basic validity filter
        .filter(F.col("pickup_datetime").isNotNull() & 
               F.col("dropoff_datetime").isNotNull() &
               F.col("trip_miles").isNotNull() &
               F.col("trip_time").isNotNull() &
               (F.col("trip_time") > 0))
    )
    
    valid_count = uber_analysis.count()
    print(f"Valid trips: {valid_count:,}")
    
    if sample_fraction is not None and sample_fraction < 1.0:
        uber_analysis = uber_analysis.sample(fraction=sample_fraction, seed=42)
        print(f"Sampled to: {uber_analysis.count():,} trips")
    
    return uber_analysis

In [ ]:
def explore_outliers_before_filtering(df, sample_size=100000):
    """
    Perform exploratory analysis on trip_miles and duration to understand outlier patterns.
    
    Args:
        df (DataFrame): Raw Uber trip data
        sample_size (int): Sample size for analysis (to avoid computing on full dataset)
    
    Returns:
        dict: Analysis results with statistics and recommendations
    """
    print("=== EXPLORATORY OUTLIER ANALYSIS ===")
    
    # Sample data for faster analysis
    if df.count() > sample_size:
        df_sample = df.sample(fraction=sample_size/df.count(), seed=42)
        print(f"Analyzing sample of {sample_size:,} trips")
    else:
        df_sample = df
        print(f"Analyzing full dataset of {df.count():,} trips")
    
    # Calculate duration in seconds
    df_analysis = df_sample.withColumn(
        "duration_sec", 
        (F.col("dropoff_datetime").cast("long") - F.col("pickup_datetime").cast("long"))
    ).filter(F.col("duration_sec").isNotNull() & F.col("trip_miles").isNotNull())
    
    results = {}
    
    for col_name in ["trip_miles", "duration_sec"]:
        print(f"\n--- {col_name.upper()} ANALYSIS ---")
        
        # Basic statistics
        stats = df_analysis.agg(
            F.count(col_name).alias("count"),
            F.mean(col_name).alias("mean"),
            F.stddev(col_name).alias("std"),
            F.min(col_name).alias("min"),
            F.max(col_name).alias("max")
        ).collect()[0]
        
        # Percentiles
        percentiles = df_analysis.approxQuantile(col_name, 
            [0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.999], 0.01)
        
        print(f"Count: {stats['count']:,}")
        print(f"Mean: {stats['mean']:.2f}")
        print(f"Std: {stats['std']:.2f}")
        print(f"Min: {stats['min']:.2f}")
        print(f"Max: {stats['max']:.2f}")
        print(f"Percentiles: 1%={percentiles[0]:.2f}, 5%={percentiles[1]:.2f}, 95%={percentiles[7]:.2f}, 99%={percentiles[8]:.2f}, 99.9%={percentiles[9]:.2f}")
        
        # Count extreme values
        if col_name == "trip_miles":
            extreme_counts = df_analysis.agg(
                F.sum(F.when(F.col(col_name) > 100, 1).otherwise(0)).alias("over_100_miles"),
                F.sum(F.when(F.col(col_name) > 200, 1).otherwise(0)).alias("over_200_miles"),
                F.sum(F.when(F.col(col_name) > 500, 1).otherwise(0)).alias("over_500_miles"),
                F.sum(F.when(F.col(col_name) < 0, 1).otherwise(0)).alias("negative"),
                F.sum(F.when(F.col(col_name) == 0, 1).otherwise(0)).alias("zero_miles")
            ).collect()[0]
            
            print(f"Trips > 100 miles: {extreme_counts['over_100_miles']:,}")
            print(f"Trips > 200 miles: {extreme_counts['over_200_miles']:,}")
            print(f"Trips > 500 miles: {extreme_counts['over_500_miles']:,}")
            print(f"Negative distances: {extreme_counts['negative']:,}")
            print(f"Zero distances: {extreme_counts['zero_miles']:,}")
            
        elif col_name == "duration_sec":
            # Convert to hours for readability
            extreme_counts = df_analysis.agg(
                F.sum(F.when(F.col(col_name) > 6*3600, 1).otherwise(0)).alias("over_6_hours"),
                F.sum(F.when(F.col(col_name) > 12*3600, 1).otherwise(0)).alias("over_12_hours"),
                F.sum(F.when(F.col(col_name) > 24*3600, 1).otherwise(0)).alias("over_24_hours"),
                F.sum(F.when(F.col(col_name) < 60, 1).otherwise(0)).alias("under_1_min"),
                F.sum(F.when(F.col(col_name) <= 0, 1).otherwise(0)).alias("zero_or_negative")
            ).collect()[0]
            
            print(f"Trips > 6 hours: {extreme_counts['over_6_hours']:,}")
            print(f"Trips > 12 hours: {extreme_counts['over_12_hours']:,}")
            print(f"Trips > 24 hours: {extreme_counts['over_24_hours']:,}")
            print(f"Trips < 1 minute: {extreme_counts['under_1_min']:,}")
            print(f"Zero/negative duration: {extreme_counts['zero_or_negative']:,}")
        
        results[col_name] = {
            "stats": stats,
            "percentiles": dict(zip([1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9], percentiles))
        }
    
    # Speed analysis
    print(f"\n--- SPEED ANALYSIS ---")
    df_speed = df_analysis.withColumn(
        "mph", 
        F.when(F.col("duration_sec") > 0, F.col("trip_miles") / (F.col("duration_sec") / 3600.0))
    ).filter(F.col("mph").isNotNull())
    
    speed_stats = df_speed.agg(
        F.mean("mph").alias("mean_mph"),
        F.stddev("mph").alias("std_mph"),
        F.max("mph").alias("max_mph")
    ).collect()[0]
    
    speed_percentiles = df_speed.approxQuantile("mph", [0.95, 0.99, 0.999], 0.01)
    
    speed_extremes = df_speed.agg(
        F.sum(F.when(F.col("mph") > 50, 1).otherwise(0)).alias("over_50_mph"),
        F.sum(F.when(F.col("mph") > 100, 1).otherwise(0)).alias("over_100_mph"),
        F.sum(F.when(F.col("mph") > 200, 1).otherwise(0)).alias("over_200_mph")
    ).collect()[0]
    
    print(f"Mean speed: {speed_stats['mean_mph']:.2f} mph")
    print(f"95th percentile: {speed_percentiles[0]:.2f} mph")
    print(f"99th percentile: {speed_percentiles[1]:.2f} mph")
    print(f"99.9th percentile: {speed_percentiles[2]:.2f} mph")
    print(f"Max speed: {speed_stats['max_mph']:.2f} mph")
    print(f"Trips > 50 mph: {speed_extremes['over_50_mph']:,}")
    print(f"Trips > 100 mph: {speed_extremes['over_100_mph']:,}")
    print(f"Trips > 200 mph: {speed_extremes['over_200_mph']:,}")
    
    return results


In [ ]:
# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10


In [ ]:
def calculate_statistical_thresholds(df, columns_config):
    """
    Calculate statistical outlier thresholds using IQR method or percentile-based approach.
    
    Args:
        df (DataFrame): Input Spark DataFrame
        columns_config (dict): Configuration for each column with method and parameters
        
    Returns:
        dict: Dictionary with calculated thresholds for each column
        
    Example:
        config = {
            "trip_miles": {"method": "iqr", "factor": 1.5},
            "duration_sec": {"method": "percentile", "lower": 0.1, "upper": 99.9}
        }
    """
    thresholds = {}
    
    for col_name, config in columns_config.items():
        if config["method"] == "iqr":
            # IQR-based outlier detection
            quantiles = df.approxQuantile(col_name, [0.25, 0.75], 0.01)  # 1% error tolerance
            if len(quantiles) == 2:
                q1, q3 = quantiles
                iqr = q3 - q1
                factor = config.get("factor", 1.5)
                lower = q1 - factor * iqr
                upper = q3 + factor * iqr
                thresholds[col_name] = {"lower": max(0, lower), "upper": upper}  # Don't go below 0
                
        elif config["method"] == "percentile":
            # Percentile-based outlier detection
            lower_pct = config.get("lower", 0.5)  # Default 0.5th percentile
            upper_pct = config.get("upper", 99.5)  # Default 99.5th percentile
            quantiles = df.approxQuantile(col_name, [lower_pct/100, upper_pct/100], 0.01)
            if len(quantiles) == 2:
                thresholds[col_name] = {"lower": max(0, quantiles[0]), "upper": quantiles[1]}
                
        elif config["method"] == "z_score":
            # Z-score based (requires computing mean and std)
            stats = df.agg(
                F.mean(col_name).alias("mean"),
                F.stddev(col_name).alias("std")
            ).collect()[0]
            
            if stats["std"] is not None and stats["std"] > 0:
                z_threshold = config.get("z_threshold", 3.0)
                mean_val = stats["mean"]
                std_val = stats["std"]
                lower = mean_val - z_threshold * std_val
                upper = mean_val + z_threshold * std_val
                thresholds[col_name] = {"lower": max(0, lower), "upper": upper}
    
    return thresholds


In [ ]:
def determine_optimal_thresholds(training_data, percentile_config=None, save_results=True, run_visualization=True):
    """
    Analyze training data to determine optimal statistical thresholds for filtering.
    SIMPLIFIED VERSION: Focuses only on statistical analysis without comparisons.
    
    Args:
        training_data (DataFrame): Uber trips from training period (e.g., 2023 data)
        percentile_config (dict): Configuration for percentile-based thresholds
        save_results (bool): Whether to save threshold results to file
        run_visualization (bool): Whether to run memory-intensive visualizations
    
    Returns:
        dict: Optimal thresholds and analysis results
    """
    print("=== DETERMINING OPTIMAL THRESHOLDS FROM TRAINING DATA ===")
    
    if percentile_config is None:
        # Conservative defaults - adjust based on your business requirements
        percentile_config = {
            "trip_miles": {"method": "percentile", "lower": 0.1, "upper": 99.5},      # Keep 99.4%
            "duration_sec": {"method": "percentile", "lower": 0.1, "upper": 99.8}     # Keep 99.7%
        }
    
    print("Configuration:")
    for field, config in percentile_config.items():
        if config["method"] == "percentile":
            print(f"  {field}: {config['lower']}% - {config['upper']}% percentile")
    
    # Calculate thresholds
    print("\n🔢 Calculating statistical thresholds...")
    thresholds = calculate_statistical_thresholds(training_data, percentile_config)
    
    # Optional visualization (memory-intensive)
    viz_results = None
    if run_visualization:
        try:
            print("\n📊 Running exploratory analysis on training data...")
            viz_results = visualize_outlier_analysis(training_data, sample_size=10000, save_plots=True)
        except Exception as e:
            print(f"⚠️  Visualization failed (memory issue): {e}")
            print("Continuing without visualization...")
            viz_results = {"error": "visualization_failed", "message": str(e)}
    else:
        print("\n⏩ Skipping visualization (set run_visualization=True to enable)")
    
    # Display calculated thresholds
    print("\n=== CALCULATED OPTIMAL THRESHOLDS ===")
    for field, thresh in thresholds.items():
        if field == "trip_miles":
            print(f"Trip Miles: {thresh['lower']:.2f} - {thresh['upper']:.2f} miles")
        elif field == "duration_sec":
            print(f"Duration: {thresh['lower']/3600:.3f} - {thresh['upper']/3600:.2f} hours")
    
    # Calculate data retention with statistical thresholds
    print("\n📊 Calculating data retention with statistical thresholds...")
    total_count = training_data.count()
    
    # Use smaller sample for memory safety
    sample_size = min(50000, total_count)
    sample_df = training_data.limit(sample_size).toPandas()
    
    print(f"Using sample of {len(sample_df):,} trips for analysis")
    
    # Statistical approach retention
    stat_valid = sample_df[
        (sample_df['trip_miles'] >= thresholds['trip_miles']['lower']) & 
        (sample_df['trip_miles'] <= thresholds['trip_miles']['upper']) &
        (sample_df['duration_sec'] >= thresholds['duration_sec']['lower']) & 
        (sample_df['duration_sec'] <= thresholds['duration_sec']['upper'])
    ]
    stat_retention = len(stat_valid) / len(sample_df) * 100
    
    print(f"\n=== DATA RETENTION ANALYSIS ===")
    print(f"Statistical approach: {stat_retention:.2f}% data retained")
    print(f"Trips filtered out: {100-stat_retention:.2f}%")
    
    # Show what gets filtered by statistical thresholds
    print(f"\n📋 FILTERING ANALYSIS (from sample):")
    miles_filtered = (sample_df['trip_miles'] < thresholds['trip_miles']['lower']) | (sample_df['trip_miles'] > thresholds['trip_miles']['upper'])
    duration_filtered = (sample_df['duration_sec'] < thresholds['duration_sec']['lower']) | (sample_df['duration_sec'] > thresholds['duration_sec']['upper'])
    
    print(f"Trips filtered by miles threshold: {miles_filtered.sum():,} ({miles_filtered.sum()/len(sample_df)*100:.2f}%)")
    print(f"Trips filtered by duration threshold: {duration_filtered.sum():,} ({duration_filtered.sum()/len(sample_df)*100:.2f}%)")
    print(f"Total trips filtered: {(miles_filtered | duration_filtered).sum():,} ({(miles_filtered | duration_filtered).sum()/len(sample_df)*100:.2f}%)")
    
    # Show extreme values that get filtered
    extreme_miles_high = sample_df['trip_miles'] > thresholds['trip_miles']['upper']
    extreme_miles_low = sample_df['trip_miles'] < thresholds['trip_miles']['lower']
    extreme_duration_high = sample_df['duration_sec'] > thresholds['duration_sec']['upper']
    extreme_duration_low = sample_df['duration_sec'] < thresholds['duration_sec']['lower']
    
    print(f"\n🔍 EXTREME VALUES FILTERED:")
    print(f"Very long trips (> {thresholds['trip_miles']['upper']:.1f} miles): {extreme_miles_high.sum():,}")
    print(f"Very short trips (< {thresholds['trip_miles']['lower']:.1f} miles): {extreme_miles_low.sum():,}")
    print(f"Very long duration (> {thresholds['duration_sec']['upper']/3600:.1f} hours): {extreme_duration_high.sum():,}")
    print(f"Very short duration (< {thresholds['duration_sec']['lower']/60:.1f} minutes): {extreme_duration_low.sum():,}")
    
    # Prepare final results
    results = {
        'thresholds': thresholds,
        'retention_analysis': {
            'statistical_pct': stat_retention,
            'filtered_pct': 100 - stat_retention
        },
        'config_used': percentile_config,
        'training_sample_size': len(sample_df),
        'total_training_size': total_count,
        'viz_stats': viz_results
    }
    
    # Save thresholds for later use
    if save_results:
        import json
        threshold_summary = {
            'trip_miles_lower': float(thresholds['trip_miles']['lower']),
            'trip_miles_upper': float(thresholds['trip_miles']['upper']),
            'duration_sec_lower': float(thresholds['duration_sec']['lower']),
            'duration_sec_upper': float(thresholds['duration_sec']['upper']),
            'generated_from': 'training_data_analysis',
            'data_retention_pct': float(stat_retention),
            'total_training_trips': int(total_count),
            'sample_size_used': int(len(sample_df))
        }
        
        with open('optimal_thresholds.json', 'w') as f:
            json.dump(threshold_summary, f, indent=2)
        print(f"\n💾 Saved thresholds to 'optimal_thresholds.json'")
    
    return results

In [ ]:
def visualize_outlier_analysis(df, sample_size=50000, save_plots=True):
    """
    Create comprehensive visualizations for outlier analysis before filtering.
    PURE DATA-DRIVEN VERSION: No hard-coded threshold comparisons.
    
    Args:
        df (DataFrame): Raw Uber trip data (Spark DataFrame)
        sample_size (int): Sample size for visualization (to avoid memory issues)
        save_plots (bool): Whether to save plots to files
    
    Returns:
        dict: Analysis results with statistics
    """
    print("=== CREATING OUTLIER VISUALIZATIONS ===")
    
    # Sample data and convert to Pandas for plotting
    if df.count() > sample_size:
        df_sample = df.sample(fraction=sample_size/df.count(), seed=42)
        print(f"Visualizing sample of {sample_size:,} trips")
    else:
        df_sample = df
        print(f"Visualizing full dataset of {df.count():,} trips")
    
    # Use existing trip_time field instead of calculating duration
    df_analysis = df_sample.withColumn(
        "duration_sec", F.col("trip_time")  # Use existing field!
    ).withColumn(
        "duration_hours", F.col("trip_time") / 3600.0
    ).withColumn(
        "mph", F.when(F.col("trip_time") > 0, F.col("trip_miles") / (F.col("trip_time") / 3600.0))
    ).filter(
        F.col("trip_time").isNotNull() & 
        F.col("trip_miles").isNotNull() &
        (F.col("trip_time") > 0)
    )
    
    # Convert to Pandas for visualization
    pdf = df_analysis.select(
        "trip_miles", "duration_sec", "duration_hours", "mph"
    ).toPandas()
    
    print(f"Collected {len(pdf):,} valid trips for visualization")
    
    # Create comprehensive visualizations
    fig = plt.figure(figsize=(20, 16))
    
    # 1. Trip Miles Distribution (Full Range)
    plt.subplot(3, 4, 1)
    plt.hist(pdf['trip_miles'], bins=100, alpha=0.7, color='skyblue', edgecolor='black')
    plt.xlabel('Trip Miles')
    plt.ylabel('Frequency')
    plt.title('Distribution of Trip Miles\n(Full Range)')
    # Show key percentiles instead of hard-coded limits
    p95 = np.percentile(pdf['trip_miles'], 95)
    p99 = np.percentile(pdf['trip_miles'], 99)
    plt.axvline(p95, color='orange', linestyle='-', alpha=0.7, label=f'95th percentile ({p95:.1f})')
    plt.axvline(p99, color='red', linestyle='-', alpha=0.7, label=f'99th percentile ({p99:.1f})')
    plt.legend()
    
    # 2. Trip Miles Distribution (Zoomed to 95th percentile)
    plt.subplot(3, 4, 2)
    miles_p95 = pdf[pdf['trip_miles'] <= p95]['trip_miles']
    plt.hist(miles_p95, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
    plt.xlabel('Trip Miles')
    plt.ylabel('Frequency')
    plt.title(f'Trip Miles Distribution\n(0 - {p95:.1f} miles, 95th percentile)')
    
    # 3. Trip Duration Distribution (Full Range)
    plt.subplot(3, 4, 3)
    plt.hist(pdf['duration_hours'], bins=100, alpha=0.7, color='coral', edgecolor='black')
    plt.xlabel('Duration (Hours)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Trip Duration\n(Full Range)')
    # Show key percentiles
    dur_p95 = np.percentile(pdf['duration_hours'], 95)
    dur_p99 = np.percentile(pdf['duration_hours'], 99)
    plt.axvline(dur_p95, color='orange', linestyle='-', alpha=0.7, label=f'95th percentile ({dur_p95:.1f}h)')
    plt.axvline(dur_p99, color='red', linestyle='-', alpha=0.7, label=f'99th percentile ({dur_p99:.1f}h)')
    plt.legend()
    
    # 4. Trip Duration Distribution (Zoomed to 95th percentile)
    plt.subplot(3, 4, 4)
    duration_p95 = pdf[pdf['duration_hours'] <= dur_p95]['duration_hours']
    plt.hist(duration_p95, bins=50, alpha=0.7, color='gold', edgecolor='black')
    plt.xlabel('Duration (Hours)')
    plt.ylabel('Frequency')
    plt.title(f'Trip Duration Distribution\n(0 - {dur_p95:.1f} hours, 95th percentile)')
    
    # 5. Speed Distribution
    plt.subplot(3, 4, 5)
    # Focus on reasonable speed range for visualization
    speed_p99 = np.percentile(pdf['mph'], 99)
    speed_filtered = pdf[(pdf['mph'] > 0) & (pdf['mph'] <= speed_p99)]['mph']
    plt.hist(speed_filtered, bins=50, alpha=0.7, color='plum', edgecolor='black')
    plt.xlabel('Average Speed (MPH)')
    plt.ylabel('Frequency')
    plt.title(f'Speed Distribution\n(0 - {speed_p99:.1f} mph, 99th percentile)')
    # Show speed percentiles
    speed_p95 = np.percentile(pdf['mph'], 95)
    plt.axvline(speed_p95, color='orange', linestyle='-', alpha=0.7, label=f'95th percentile ({speed_p95:.1f})')
    plt.legend()
    
    # 6. Box Plot - Trip Miles
    plt.subplot(3, 4, 6)
    box_data_miles = pdf[pdf['trip_miles'] <= p95]['trip_miles']  # Use data-driven limit
    plt.boxplot(box_data_miles, vert=True, patch_artist=True, 
                boxprops=dict(facecolor='lightblue', alpha=0.7))
    plt.ylabel('Trip Miles')
    plt.title(f'Trip Miles Box Plot\n(0 - {p95:.1f} miles)')
    plt.grid(True, alpha=0.3)
    
    # 7. Box Plot - Duration
    plt.subplot(3, 4, 7)
    box_data_duration = pdf[pdf['duration_hours'] <= dur_p95]['duration_hours']
    plt.boxplot(box_data_duration, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightcoral', alpha=0.7))
    plt.ylabel('Duration (Hours)')
    plt.title(f'Duration Box Plot\n(0 - {dur_p95:.1f} hours)')
    plt.grid(True, alpha=0.3)
    
    # 8. Scatter Plot - Miles vs Duration
    plt.subplot(3, 4, 8)
    scatter_data = pdf[(pdf['trip_miles'] <= p95) & (pdf['duration_hours'] <= dur_p95)]
    plt.scatter(scatter_data['duration_hours'], scatter_data['trip_miles'], 
                alpha=0.5, s=1, c='purple')
    plt.xlabel('Duration (Hours)')
    plt.ylabel('Trip Miles')
    plt.title('Miles vs Duration\n(95th percentile data)')
    plt.grid(True, alpha=0.3)
    
    # 9. Percentile Analysis - Miles
    plt.subplot(3, 4, 9)
    percentiles = np.arange(90, 100, 0.1)
    miles_percentiles = [np.percentile(pdf['trip_miles'], p) for p in percentiles]
    plt.plot(percentiles, miles_percentiles, 'b-', linewidth=2)
    plt.xlabel('Percentile')
    plt.ylabel('Trip Miles')
    plt.title('High Percentiles\nTrip Miles (90-100%)')
    plt.grid(True, alpha=0.3)
    
    # 10. Percentile Analysis - Duration
    plt.subplot(3, 4, 10)
    duration_percentiles = [np.percentile(pdf['duration_hours'], p) for p in percentiles]
    plt.plot(percentiles, duration_percentiles, 'r-', linewidth=2)
    plt.xlabel('Percentile')
    plt.ylabel('Duration (Hours)')
    plt.title('High Percentiles\nDuration (90-100%)')
    plt.grid(True, alpha=0.3)
    
    # 11. Data Quality Indicators
    plt.subplot(3, 4, 11)
    quality_stats = {
        'Very Short\n(< 0.5 mi)': len(pdf[pdf['trip_miles'] < 0.5]),
        'Very Long\n(> 99%ile)': len(pdf[pdf['trip_miles'] > np.percentile(pdf['trip_miles'], 99)]),
        'Quick Trips\n(< 2 min)': len(pdf[pdf['duration_hours'] < 2/60]),
        'Long Trips\n(> 99%ile)': len(pdf[pdf['duration_hours'] > np.percentile(pdf['duration_hours'], 99)]),
        'High Speed\n(> 95%ile)': len(pdf[pdf['mph'] > np.percentile(pdf['mph'], 95)]),
        'Zero Distance': len(pdf[pdf['trip_miles'] == 0])
    }
    
    colors = ['lightblue', 'blue', 'lightcoral', 'red', 'lightgreen', 'orange']
    bars = plt.bar(range(len(quality_stats)), list(quality_stats.values()), 
                   color=colors, alpha=0.7, edgecolor='black')
    plt.xticks(range(len(quality_stats)), list(quality_stats.keys()), rotation=45, ha='right')
    plt.ylabel('Count')
    plt.title('Data Quality Indicators')
    plt.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for i, bar in enumerate(bars):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                f'{int(height):,}', ha='center', va='bottom', fontsize=8)
    
    # 12. Statistical Summary Table
    plt.subplot(3, 4, 12)
    plt.axis('off')
    
    # Calculate key statistics
    stats_summary = f'''
    DATA-DRIVEN STATISTICS
    
    Trip Miles:
    • Mean: {pdf['trip_miles'].mean():.2f}
    • Median: {pdf['trip_miles'].median():.2f}
    • 90th %tile: {np.percentile(pdf['trip_miles'], 90):.2f}
    • 95th %tile: {np.percentile(pdf['trip_miles'], 95):.2f}
    • 99th %tile: {np.percentile(pdf['trip_miles'], 99):.2f}
    • 99.5th %tile: {np.percentile(pdf['trip_miles'], 99.5):.2f}
    
    Duration (hours):
    • Mean: {pdf['duration_hours'].mean():.2f}
    • Median: {pdf['duration_hours'].median():.2f}
    • 90th %tile: {np.percentile(pdf['duration_hours'], 90):.2f}
    • 95th %tile: {np.percentile(pdf['duration_hours'], 95):.2f}
    • 99th %tile: {np.percentile(pdf['duration_hours'], 99):.2f}
    • 99.8th %tile: {np.percentile(pdf['duration_hours'], 99.8):.2f}
    
    Speed (mph):
    • Mean: {pdf['mph'].mean():.2f}
    • 90th %tile: {np.percentile(pdf['mph'], 90):.2f}
    • 95th %tile: {np.percentile(pdf['mph'], 95):.2f}
    • 99th %tile: {np.percentile(pdf['mph'], 99):.2f}
    '''
    
    plt.text(0.05, 0.95, stats_summary, transform=plt.gca().transAxes, 
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
    
    plt.tight_layout()
    
    if save_plots:
        plt.savefig('outlier_analysis_data_driven.png', dpi=300, bbox_inches='tight')
        print("📊 Saved data-driven analysis to 'outlier_analysis_data_driven.png'")
    
    plt.show()
    
    # Return key statistics for programmatic use
    return {
        'trip_miles': {
            'mean': pdf['trip_miles'].mean(),
            'median': pdf['trip_miles'].median(),
            'percentiles': {p: np.percentile(pdf['trip_miles'], p) for p in [90, 95, 99, 99.5, 99.9]}
        },
        'duration_hours': {
            'mean': pdf['duration_hours'].mean(),
            'median': pdf['duration_hours'].median(),
            'percentiles': {p: np.percentile(pdf['duration_hours'], p) for p in [90, 95, 99, 99.5, 99.8]}
        },
        'speed_mph': {
            'mean': pdf['mph'].mean(),
            'percentiles': {p: np.percentile(pdf['mph'], p) for p in [90, 95, 99]}
        },
        'total_sample_size': len(pdf)
    }

In [ ]:
# ===================================================================
# COMPLETE EXPLORATORY ANALYSIS & OUTLIER DETECTION PIPELINE
# ===================================================================

# Step 1: Extract Uber data from training period (2023)
print("=== STEP 1: EXTRACTING TRAINING DATA ===")
training_data = extract_uber_data_for_analysis(
    year_list=[2023],           # Use 2023 as training
    month_list=None,            # All months
    sample_fraction=0.1        # Use all data for robust thresholds
)

# Step 2: Run comprehensive analysis and determine optimal thresholds
print("\n=== STEP 2: COMPREHENSIVE ANALYSIS & THRESHOLD DETERMINATION ===")
threshold_results = determine_optimal_thresholds(
    training_data, 
    percentile_config={
        "trip_miles": {"method": "percentile", "lower": 0.1, "upper": 99.5},     # Keep 99.4% of data
        "duration_sec": {"method": "percentile", "lower": 0.1, "upper": 99.8}    # Keep 99.7% of data
    },
    save_results=True
)

# Step 3: Display key results
print("\n=== STEP 3: KEY RESULTS SUMMARY ===")
print(f"🎯 Optimal Thresholds Determined:")
print(f"   • Trip Miles: {threshold_results['thresholds']['trip_miles']['lower']:.2f} - {threshold_results['thresholds']['trip_miles']['upper']:.2f} miles")
print(f"   • Duration: {threshold_results['thresholds']['duration_sec']['lower']/3600:.3f} - {threshold_results['thresholds']['duration_sec']['upper']/3600:.2f} hours")

# Replace the problematic print statement with this:

print(f"\n=== DATA RETENTION ANALYSIS ===")
if 'retention_analysis' in threshold_results:
    retention = threshold_results['retention_analysis']
    
    # Check which keys actually exist
    if 'statistical_pct' in retention:
        print(f"Statistical approach: {retention['statistical_pct']:.2f}% data retained")
    
    if 'filtered_pct' in retention:
        print(f"Trips filtered out: {retention['filtered_pct']:.2f}%")
    
    # Only print difference if it exists
    if 'difference_pct' in retention:
        print(f"Improvement: {retention['difference_pct']:+.2f}%")
    else:
        print("Difference calculation not available")
else:
    print("Retention analysis not available")

print(f"\n📁 Files Generated:")
print(f"   • optimal_thresholds.json (thresholds for production)")
if threshold_results.get('viz_stats') and 'error' not in threshold_results['viz_stats']:
    print(f"   • outlier_analysis_data_driven.png (visualization)")


### External Data: MTA CBD Data

In [ ]:
def build_cbd_locationids_csv_if_needed(
    zones_geojson: str,
    cbd_geojson: str,
    out_csv: str,
    planar_crs: int = 2263,
    overlap_ratio_thresh: float = 0.25,
):
    """
    Create a CSV mapping LocationID -> in_cbd (0/1) by intersecting TLC zones with CBD polygons.
    This runs on the driver using GeoPandas/Shapely and only when out_csv is missing.
    
    Args:
        zones_geojson (str): Path to TLC taxi zones GeoJSON file
        cbd_geojson (str): Path to MTA CBD boundary GeoJSON file
        out_csv (str): Output CSV file path
        planar_crs (int): EPSG code for planar coordinate system (default: 2263 - NY Long Island)
        overlap_ratio_thresh (float): Minimum overlap ratio to mark zone as CBD (default: 0.01 = 1%)
    
    Returns:
        None: Creates CSV file with LocationID and in_cbd columns
    
    Process:
        1. Checks if output CSV already exists (skips if found)
        2. Loads TLC zones and CBD boundaries using GeoPandas
        3. Projects to planar CRS for accurate area calculations
        4. Calculates spatial intersection and overlap ratios
        5. Marks zones as CBD if overlap > threshold
        6. Saves results as CSV
    """
    import os
    
    if os.path.exists(out_csv):
        print(f"[CBD] Mapping exists, skipping: {out_csv}")
        return

    try:
        import geopandas as gpd
    except Exception as e:
        raise RuntimeError(
            "GeoPandas is required to build the CBD mapping. "
            "Install with: pip install geopandas shapely pyproj fiona"
        ) from e

    print(f"[CBD] Processing: {zones_geojson} + {cbd_geojson} -> {out_csv}")

    # Load geospatial data
    zones = gpd.read_file(zones_geojson)
    cbd   = gpd.read_file(cbd_geojson)

    # Find correct LocationID column name (case-insensitive)
    lower_map = {c.lower(): c for c in zones.columns}
    loc_col = lower_map.get("locationid", lower_map.get("location_id"))
    if loc_col is None:
        raise ValueError("Could not find LocationID/location_id in TLC zones GeoJSON.")

    print(f"[CBD] Found LocationID column: {loc_col}")
    print(f"[CBD] TLC zones: {len(zones)} zones")
    print(f"[CBD] CBD polygons: {len(cbd)} polygons")

    # Project to planar CRS for accurate area calculations
    zones_proj = zones.to_crs(planar_crs)
    cbd_proj   = cbd.to_crs(planar_crs)

    # Create union of all CBD polygons
    cbd_union = cbd_proj.unary_union

    # Calculate overlap ratio: intersection_area / zone_area
    print(f"[CBD] Calculating spatial intersections...")
    intersection_area = zones_proj.geometry.intersection(cbd_union).area
    zone_area = zones_proj.geometry.area
    
    # Handle potential division by zero
    overlap_ratio = intersection_area / zone_area.replace(0, 1)  # Replace 0 with 1 to avoid division by zero
    overlap_ratio = overlap_ratio.fillna(0.0)

    # Mark zones as CBD if overlap ratio exceeds threshold
    zones_proj["overlap_ratio"] = overlap_ratio
    zones_proj["in_cbd"] = (zones_proj["overlap_ratio"] > overlap_ratio_thresh).astype(int)

    # Prepare output dataframe
    output_df = zones_proj[[loc_col, "in_cbd", "overlap_ratio"]].copy()
    output_df.rename(columns={loc_col: "LocationID"}, inplace=True)
    output_df["LocationID"] = output_df["LocationID"].astype(int)

    # Sort by LocationID for clean output
    output_df = output_df.sort_values("LocationID")

    # Save to CSV
    output_df[["LocationID", "in_cbd"]].to_csv(out_csv, index=False)

    # Summary statistics
    total_zones = len(output_df)
    cbd_zones = output_df["in_cbd"].sum()
    cbd_percentage = cbd_zones / total_zones * 100

    print(f"[CBD] Results:")
    print(f"  Total zones: {total_zones}")
    print(f"  CBD zones: {cbd_zones} ({cbd_percentage:.1f}%)")
    print(f"  Non-CBD zones: {total_zones - cbd_zones} ({100-cbd_percentage:.1f}%)")
    print(f"  Overlap threshold: {overlap_ratio_thresh*100:.1f}%")
    print(f"[CBD] Wrote mapping: {out_csv}")

    # Optional: Save detailed version with overlap ratios for debugging
    debug_csv = out_csv.replace('.csv', '_detailed.csv')
    output_df.to_csv(debug_csv, index=False)
    print(f"[CBD] Wrote detailed mapping (with overlap ratios): {debug_csv}")

In [ ]:
# Process MTA Central Business District boundaries
def process_mta_cbd_data():
    """
    Process MTA CBD GeoJSON and create LocationID mapping.
    """
    print("=== PROCESSING MTA CENTRAL BUSINESS DISTRICT DATA ===")
    
    # File paths
    TLC_ZONES_GEOJSON = "./lake/landing/lookups/taxi_zones.geojson"
    MTA_CBD_GEOJSON = "./lake/landing/lookups/MTA Central Business District Taxi Zones_20250809.geojson"
    MTA_CBD_CSV = "./lake/landing/lookups/mta_cbd_locationids.csv"
    
    # Build CBD LocationID mapping if not exists
    build_cbd_locationids_csv_if_needed(
        zones_geojson=TLC_ZONES_GEOJSON,
        cbd_geojson=MTA_CBD_GEOJSON, 
        out_csv=MTA_CBD_CSV,
        planar_crs=2263,  # NY Long Island projection
        overlap_ratio_thresh=0.25  # 25% overlap threshold
    )
    
    return MTA_CBD_CSV

# Run the processing
cbd_csv_path = process_mta_cbd_data()

In [ ]:
def load_tlc_and_cbd_dimensions_fixed():
    """
    Fixed version of the loading function with proper data type handling.
    """
    print("=== LOADING TLC AND CBD DIMENSION TABLES (FIXED) ===")
    
    # Load TLC zones
    TLC_ZONES_CSV = "./lake/landing/lookups/taxi_zone_lookup.csv"
    
    try:
        tlc_dim = (spark.read.csv(TLC_ZONES_CSV, header=True)
                   .select(F.col("LocationID").cast("int").alias("LocationID"),
                           "Borough", "Zone", "service_zone")
                   .dropna(subset=["LocationID"])
                   .dropDuplicates(["LocationID"]))
        
        print(f"✅ Loaded TLC zones: {tlc_dim.count()} locations")
        tlc_dim.groupBy("Borough").count().show()
        
    except Exception as e:
        print(f"❌ Failed to load TLC zones: {e}")
        tlc_dim = None
    
    # Load CBD mapping - FIXED VERSION
    CBD_CSV = "./lake/landing/lookups/mta_cbd_locationids.csv"
    
    try:
        cbd_dim = (spark.read.csv(CBD_CSV, header=True)
                   .select(F.col("LocationID").cast("int").alias("LocationID"),
                           F.col("in_cbd").cast("int").alias("in_cbd"))  # CAST TO INT!
                   .filter(F.col("in_cbd") == 1)  # ONLY KEEP CBD ZONES
                   .dropDuplicates(["LocationID"]))
        
        cbd_count = cbd_dim.count()
        print(f"✅ Loaded CBD mapping: {cbd_count} zones marked as CBD")
        
        # Show which boroughs have CBD zones
        if tlc_dim is not None:
            cbd_by_borough = (cbd_dim.join(tlc_dim, "LocationID")
                             .groupBy("Borough").count()
                             .orderBy(F.desc("count")))
            print("CBD zones by borough:")
            cbd_by_borough.show()
        
    except Exception as e:
        print(f"❌ Failed to load CBD mapping: {e}")
        cbd_dim = None
    
    return tlc_dim, cbd_dim

# Test the fixed version
TLC_DIM_FIXED, CBD_DIM_FIXED = load_tlc_and_cbd_dimensions_fixed()

In [ ]:
def add_trip_classification_labels(df, tlc_dim, cbd_dim):
    """
    Add comprehensive trip classification labels using TLC zones and MTA CBD data.
    
    Creates the trip type classifications we discussed:
    - intra_cbd: Both pickup and dropoff in CBD
    - inbound_cbd: Pickup outside CBD, dropoff inside CBD  
    - outbound_cbd: Pickup inside CBD, dropoff outside CBD
    - non_cbd: Both pickup and dropoff outside CBD
    
    Also adds borough and zone information.
    """
    print("=== ADDING TRIP CLASSIFICATION LABELS ===")
    
    enriched_df = df
    
    # === TLC ZONE ENRICHMENT ===
    if tlc_dim is not None:
        print("📍 Adding TLC zone information...")
        
        enriched_df = (enriched_df
            # Add pickup zone details
            .join(broadcast(tlc_dim.withColumnRenamed("LocationID","PU_ID")),
                  enriched_df.PULocationID == F.col("PU_ID"), "left")
            .withColumnRenamed("Borough","PU_Borough")
            .withColumnRenamed("Zone","PU_Zone") 
            .withColumnRenamed("service_zone","PU_service_zone")
            .drop("PU_ID")
            
            # Add dropoff zone details
            .join(broadcast(tlc_dim.withColumnRenamed("LocationID","DO_ID")),
                  enriched_df.DOLocationID == F.col("DO_ID"), "left")
            .withColumnRenamed("Borough","DO_Borough")
            .withColumnRenamed("Zone","DO_Zone")
            .withColumnRenamed("service_zone","DO_service_zone") 
            .drop("DO_ID")
        )
        
        print("✅ Added TLC zone information (boroughs, zones)")
    else:
        print("⚠️  TLC zone enrichment skipped - no TLC data available")
    
    # === CBD CLASSIFICATION ===
    if cbd_dim is not None:
        print("🏢 Adding CBD classification...")
        
        enriched_df = (enriched_df
            # Mark pickup locations in CBD
            .join(broadcast(cbd_dim.withColumnRenamed("LocationID","PU_ID")),
                  enriched_df.PULocationID == F.col("PU_ID"), "left")
            .withColumn("PU_in_cbd", F.when(F.col("in_cbd").isNotNull(), 1).otherwise(0))
            .drop("PU_ID","in_cbd")
            
            # Mark dropoff locations in CBD  
            .join(broadcast(cbd_dim.withColumnRenamed("LocationID","DO_ID")),
                  enriched_df.DOLocationID == F.col("DO_ID"), "left")
            .withColumn("DO_in_cbd", F.when(F.col("in_cbd").isNotNull(), 1).otherwise(0))
            .drop("DO_ID","in_cbd")
            
            # Create trip type classification
            .withColumn("trip_cbd_type", 
                F.when((F.col("PU_in_cbd") == 1) & (F.col("DO_in_cbd") == 1), "intra_cbd")
                 .when((F.col("PU_in_cbd") == 1) & (F.col("DO_in_cbd") == 0), "outbound_cbd") 
                 .when((F.col("PU_in_cbd") == 0) & (F.col("DO_in_cbd") == 1), "inbound_cbd")
                 .otherwise("non_cbd")
            )
        )
        
        print("✅ Added CBD classification (trip_cbd_type)")
        
        # Show classification distribution
        if enriched_df.count() < 1000000:  # Only for reasonable sample sizes
            print("\n📊 CBD Trip Type Distribution:")
            enriched_df.groupBy("trip_cbd_type").count().orderBy(F.desc("count")).show()
        
    else:
        print("⚠️  CBD classification skipped - no CBD data available")
        # Add placeholder columns for consistency
        enriched_df = (enriched_df
            .withColumn("PU_in_cbd", F.lit(0))
            .withColumn("DO_in_cbd", F.lit(0)) 
            .withColumn("trip_cbd_type", F.lit("unknown"))
        )
    
    # === ADDITIONAL USEFUL CLASSIFICATIONS ===
    enriched_df = (enriched_df
        # Cross-borough trips
        .withColumn("is_cross_borough",
            F.when((F.col("PU_Borough").isNotNull()) & (F.col("DO_Borough").isNotNull()),
                   F.when(F.col("PU_Borough") != F.col("DO_Borough"), 1).otherwise(0))
             .otherwise(F.lit(None)))
        
        # Manhattan-related trips (if TLC data available)
        .withColumn("involves_manhattan",
            F.when((F.col("PU_Borough") == "Manhattan") | (F.col("DO_Borough") == "Manhattan"), 1)
             .otherwise(0))
             
        # Airport-related trips (if service zone available)
        .withColumn("involves_airport", 
            F.when((F.col("PU_service_zone") == "Airports") | (F.col("DO_service_zone") == "Airports"), 1)
             .otherwise(0))
    )
    
    print("✅ Added additional trip classifications")
    
    return enriched_df

def analyze_trip_classifications(df):
    """
    Analyze the distribution of trip classifications.
    """
    print("=== TRIP CLASSIFICATION ANALYSIS ===")
    
    # Sample for analysis
    sample_df = df.limit(10000).toPandas()
    
    print(f"Analysis based on sample of {len(sample_df):,} trips:")
    
    if 'trip_cbd_type' in sample_df.columns:
        print(f"\n🏢 CBD Trip Types:")
        cbd_dist = sample_df['trip_cbd_type'].value_counts()
        for trip_type, count in cbd_dist.items():
            pct = count / len(sample_df) * 100
            print(f"  {trip_type}: {count:,} trips ({pct:.1f}%)")
    
    if 'PU_Borough' in sample_df.columns and 'DO_Borough' in sample_df.columns:
        print(f"\n🗽 Borough Patterns:")
        print(f"  Cross-borough trips: {sample_df['is_cross_borough'].sum():,} ({sample_df['is_cross_borough'].mean()*100:.1f}%)")
        print(f"  Manhattan-involved trips: {sample_df['involves_manhattan'].sum():,} ({sample_df['involves_manhattan'].mean()*100:.1f}%)")
    
    if 'involves_airport' in sample_df.columns:
        print(f"  Airport trips: {sample_df['involves_airport'].sum():,} ({sample_df['involves_airport'].mean()*100:.1f}%)")
    
    return sample_df

In [ ]:
def visualize_nyc_cbd_and_taxi_zones_corrected(save_plot=True, figsize=(15, 12)):
    """
    Corrected version that properly merges TLC lookup data with CBD data.
    """
    import geopandas as gpd
    import matplotlib.pyplot as plt
    import pandas as pd
    
    print("=== VISUALIZING NYC CBD AND TAXI ZONES (CORRECTED) ===")
    
    try:
        # Load the geospatial data
        zones_file = "./lake/landing/lookups/taxi_zones.geojson"
        cbd_file = "./lake/landing/lookups/MTA Central Business District Taxi Zones_20250809.geojson"
        
        print(f"📂 Loading geospatial files...")
        zones = gpd.read_file(zones_file)
        cbd_boundaries = gpd.read_file(cbd_file)
        
        print(f"✅ Loaded {len(zones)} taxi zones")
        print(f"✅ Loaded {len(cbd_boundaries)} CBD polygons")
        
        # Load the lookup tables
        print(f"📂 Loading lookup tables...")
        tlc_lookup = pd.read_csv("./lake/landing/lookups/taxi_zone_lookup.csv")
        cbd_mapping = pd.read_csv("./lake/landing/lookups/mta_cbd_locationids.csv")
        
        print(f"✅ Loaded TLC lookup: {len(tlc_lookup)} zones with Borough data")
        print(f"✅ Loaded CBD mapping: {len(cbd_mapping)} zones with CBD classification")
        
        # Find LocationID column in GeoJSON (case-insensitive)
        location_cols = [col for col in zones.columns if 'location' in col.lower()]
        if location_cols:
            locationid_col = location_cols[0]
            print(f"📍 Using LocationID column: {locationid_col}")
        else:
            raise ValueError("No LocationID column found in zones GeoJSON")
        
        # === PROPER MERGE SEQUENCE ===
        # Step 1: Merge zones with TLC lookup (to get Borough, Zone, service_zone)
        zones_with_tlc = zones.merge(
            tlc_lookup, 
            left_on=locationid_col, 
            right_on='LocationID', 
            how='left'
        )
        print(f"✅ Merged zones with TLC lookup")
        
        # Step 2: Merge with CBD mapping (to get in_cbd status)
        zones_merged = zones_with_tlc.merge(
            cbd_mapping, 
            on='LocationID',  # Both have LocationID now
            how='left'
        )
        zones_merged['in_cbd'] = zones_merged['in_cbd'].fillna(0).astype(int)
        print(f"✅ Merged with CBD classification")
        
        # Debug: Show final column structure
        print(f"🔍 Final columns: {list(zones_merged.columns)}")
        print(f"📊 Borough distribution:")
        print(zones_merged['Borough'].value_counts())
        
        # Set up the plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
        
        # === PLOT 1: ZONES COLORED BY CBD STATUS ===
        ax1.set_title('NYC Taxi Zones: CBD Classification\n(38 zones, 14.6%)', fontsize=14, fontweight='bold')
        
        # Plot non-CBD zones
        non_cbd = zones_merged[zones_merged['in_cbd'] == 0]
        non_cbd.plot(ax=ax1, color='lightgray', alpha=0.6, edgecolor='white', linewidth=0.5, label=f'Non-CBD ({len(non_cbd)})')
        
        # Plot CBD zones
        cbd_zones = zones_merged[zones_merged['in_cbd'] == 1]
        cbd_zones.plot(ax=ax1, color='red', alpha=0.8, edgecolor='darkred', linewidth=0.8, label=f'CBD Zones ({len(cbd_zones)})')
        
        # Overlay CBD boundaries
        cbd_boundaries.plot(ax=ax1, color='none', edgecolor='blue', linewidth=2.5, alpha=0.9, label='MTA CBD Boundary')
        
        ax1.set_xlabel('Longitude')
        ax1.set_ylabel('Latitude')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # === PLOT 2: ZONES BY BOROUGH ===
        ax2.set_title('NYC Taxi Zones: By Borough', fontsize=14, fontweight='bold')
        
        # Define colors for boroughs
        borough_colors = {
            'Manhattan': '#FF6B6B',    # Red
            'Brooklyn': '#4ECDC4',     # Teal
            'Queens': '#45B7D1',       # Blue
            'Bronx': '#F9CA24',        # Yellow
            'Staten Island': '#F0932B', # Orange
            'EWR': '#6C5CE7',          # Purple
            'Unknown': '#A4A4A4',      # Gray
            'N/A': '#D3D3D3'           # Light gray
        }
        
        # Plot zones by borough
        for borough in zones_merged['Borough'].unique():
            if pd.notna(borough):
                borough_zones = zones_merged[zones_merged['Borough'] == borough]
                color = borough_colors.get(borough, '#808080')  # Default gray
                borough_zones.plot(ax=ax2, color=color, alpha=0.7, edgecolor='white', 
                                 linewidth=0.5, label=f'{borough} ({len(borough_zones)})')
        
        ax2.set_xlabel('Longitude')
        ax2.set_ylabel('Latitude')
        ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        # Save the plot
        if save_plot:
            output_file = "nyc_cbd_taxi_zones_analysis_corrected.png"
            plt.savefig(output_file, dpi=300, bbox_inches='tight')
            print(f"💾 Saved visualization: {output_file}")
        
        plt.show()
        
        # === DETAILED SUMMARY STATISTICS ===
        total_zones = len(zones_merged)
        cbd_zones_count = len(zones_merged[zones_merged['in_cbd'] == 1])
        cbd_percentage = cbd_zones_count / total_zones * 100
        
        print(f"\n📊 OVERALL SUMMARY:")
        print(f"  Total taxi zones: {total_zones}")
        print(f"  CBD zones: {cbd_zones_count} ({cbd_percentage:.1f}%)")
        print(f"  Non-CBD zones: {total_zones - cbd_zones_count} ({100-cbd_percentage:.1f}%)")
        
        print(f"\n🏛️ CBD ZONES BY BOROUGH:")
        cbd_by_borough = (zones_merged[zones_merged['in_cbd'] == 1]
                         .groupby('Borough')
                         .size()
                         .sort_values(ascending=False))
        
        for borough, count in cbd_by_borough.items():
            total_in_borough = len(zones_merged[zones_merged['Borough'] == borough])
            pct = count / total_in_borough * 100
            print(f"  {borough}: {count}/{total_in_borough} zones ({pct:.1f}% of borough)")
        
        print(f"\n🗽 ALL ZONES BY BOROUGH:")
        borough_summary = zones_merged.groupby('Borough').agg({
            'in_cbd': ['count', 'sum']
        })
        borough_summary.columns = ['Total_Zones', 'CBD_Zones']
        borough_summary['CBD_Percentage'] = (borough_summary['CBD_Zones'] / borough_summary['Total_Zones'] * 100).round(1)
        print(borough_summary.sort_values('CBD_Zones', ascending=False))
        
        return zones_merged, cbd_boundaries
        
    except Exception as e:
        print(f"❌ Visualization failed: {e}")
        import traceback
        print(f"Traceback: {traceback.format_exc()}")
        return None, None

# Run the corrected visualization
print("🚀 Running corrected visualization with proper data merging...")
zones_final, cbd_final = visualize_nyc_cbd_and_taxi_zones_corrected()

In [ ]:
# SIMPLE MTA + TLC TEST
print("=== SIMPLE MTA + TLC SPATIAL FEATURES TEST ===")

# Step 1: Setup dimensions
cbd_csv_path = process_mta_cbd_data()
TLC_DIM, CBD_DIM = load_tlc_and_cbd_dimensions_fixed()

# Step 2: Load small sample
sample_data = extract_uber_data_for_analysis(
    year_list=[2023],
    month_list=[1], 
    sample_fraction=0.01  # 1% sample
)

# Step 3: Add spatial features
enriched_data = add_trip_classification_labels(sample_data, TLC_DIM, CBD_DIM)

# Step 4: Check results
print(f"✅ Added spatial features to {enriched_data.count():,} trips")
print(f"📊 Total columns: {len(enriched_data.columns)}")

# Show CBD trip distribution
enriched_data.groupBy("trip_cbd_type").count().show()

print("🎯 Spatial features working! Ready for additional external data.")

In [ ]:
def create_monthly_spatial_features_and_save():
    """
    Create monthly spatial features based on the MTA CBD data and save them.
    Reproduce the spatial feature set defined in the External Data: MTA CBD section.
    """
    
    print("🗺️ CREATING AND SAVING MONTHLY SPATIAL FEATURES")
    print("Based on actual MTA CBD + TLC processing code")
    print("="*70)
    
    # === Spatial features created (from the code) ===
    actual_spatial_features = [
        # CBD classification features
        'PU_in_cbd',            # 1: Pickup in CBD (0/1)
        'DO_in_cbd',            # 2: Dropoff in CBD (0/1)
        'trip_cbd_type',        # 3: Trip CBD type (intra_cbd, inbound_cbd, outbound_cbd, non_cbd)
        
        # TLC Zone information
        'PU_Borough',           # 4: Pickup borough name
        'DO_Borough',           # 5: Dropoff borough name
        'PU_Zone',              # 6: Pickup zone name
        'DO_Zone',              # 7: Dropoff zone name
        'PU_service_zone',      # 8: Pickup service zone type
        'DO_service_zone',      # 9: Dropoff service zone type
        
        # Derived spatial features
        'is_cross_borough',     # 10: Cross-borough trip (True/False)
        'involves_manhattan',   # 11: Trip involves Manhattan (True/False)
        'involves_airport'      # 12: Trip involves an airport (True/False)
    ]
    
    print(f"📋 Will create {len(actual_spatial_features)} spatial features")
    
    try:
        # === STEP 1: Recreate spatial processing functions ===
        print("\n🗺️ Step 1: Setting up spatial processing functions...")
        
        def build_cbd_locationids_csv_if_needed_recreated():
            """Recreate CBD mapping artifacts if needed"""
            
            zones_geojson = "./lake/landing/lookups/taxi_zones.geojson"
            cbd_geojson = "./lake/landing/lookups/MTA Central Business District Taxi Zones_20250809.geojson"
            out_csv = "./lake/landing/lookups/mta_cbd_locationids.csv"
            
            import os
            if os.path.exists(out_csv):
                print(f"[CBD] Mapping exists, using existing: {out_csv}")
                return out_csv
            
            try:
                import geopandas as gpd
                
                print(f"[CBD] Processing: {zones_geojson} + {cbd_geojson} -> {out_csv}")
                
                # Load geospatial data
                zones = gpd.read_file(zones_geojson)
                cbd = gpd.read_file(cbd_geojson)
                
                # Find LocationID column
                locationid_col = None
                for col in zones.columns:
                    if col.lower() in ['locationid', 'location_id']:
                        locationid_col = col
                        break
                
                if locationid_col is None:
                    raise ValueError("Could not find LocationID column in zones")
                
                print(f"[CBD] Found LocationID column: {locationid_col}")
                print(f"[CBD] TLC zones: {len(zones)} zones")
                print(f"[CBD] CBD polygons: {len(cbd)} polygons")
                
                # Project to planar CRS for accurate area calculations
                planar_crs = 2263  # NAD83 / New York Long Island
                zones_proj = zones.to_crs(planar_crs)
                cbd_proj = cbd.to_crs(planar_crs)
                
                # Create union of all CBD polygons
                cbd_union = cbd_proj.unary_union
                
                # Calculate overlap ratio: intersection_area / zone_area
                intersection_area = zones_proj.geometry.intersection(cbd_union).area
                zone_area = zones_proj.geometry.area
                overlap_ratio = intersection_area / zone_area.replace(0, 1)
                overlap_ratio = overlap_ratio.fillna(0.0)
                
                # Mark zones as CBD if overlap ratio exceeds threshold (1%)
                overlap_thresh = 0.01
                zones_proj["in_cbd"] = (overlap_ratio > overlap_thresh).astype(int)
                
                # Prepare output
                output_df = zones_proj[[locationid_col, "in_cbd"]].copy()
                output_df.rename(columns={locationid_col: "LocationID"}, inplace=True)
                output_df["LocationID"] = output_df["LocationID"].astype(int)
                output_df = output_df.sort_values("LocationID")
                
                # Save to CSV
                output_df.to_csv(out_csv, index=False)
                
                total_zones = len(output_df)
                cbd_zones = output_df["in_cbd"].sum()
                
                print(f"[CBD] Results:")
                print(f"  Total zones: {total_zones}")
                print(f"  CBD zones: {cbd_zones} ({cbd_zones/total_zones*100:.1f}%)")
                print(f"[CBD] Wrote mapping: {out_csv}")
                
                return out_csv
                
            except Exception as e:
                print(f"[CBD] Processing failed: {e}")
                return None
        
        def load_tlc_and_cbd_dimensions_recreated():
            """Recreate function to load TLC and CBD dimension tables"""
            
            # Load TLC zone lookup
            tlc_csv = "./lake/landing/lookups/taxi_zone_lookup.csv"
            tlc_dim = (spark.read.csv(tlc_csv, header=True)
                      .withColumn("LocationID", F.col("LocationID").cast("int")))
            
            # Load CBD mapping
            cbd_csv = "./lake/landing/lookups/mta_cbd_locationids.csv"
            cbd_dim = (spark.read.csv(cbd_csv, header=True)
                      .withColumn("LocationID", F.col("LocationID").cast("int"))
                      .filter(F.col("in_cbd") == 1))  # Only CBD zones
            
            print(f"✅ Loaded TLC zones: {tlc_dim.count()} locations")
            print(f"✅ Loaded CBD mapping: {cbd_dim.count()} CBD zones")
            
            return tlc_dim, cbd_dim
        
        def add_trip_classification_labels_recreated(df, tlc_dim, cbd_dim):
            """Recreate function that adds spatial features to a DataFrame"""
            
            print("🗺️ Adding comprehensive spatial features...")
            
            enriched_df = df
            
            # === TLC ZONE ENRICHMENT ===
            if tlc_dim is not None:
                print("📍 Adding TLC zone information...")
                
                # Add pickup zone details
                enriched_df = (enriched_df
                    .join(broadcast(tlc_dim.withColumnRenamed("LocationID","PU_ID")),
                          enriched_df.PULocationID == F.col("PU_ID"), "left")
                    .withColumnRenamed("Borough","PU_Borough")
                    .withColumnRenamed("Zone","PU_Zone") 
                    .withColumnRenamed("service_zone","PU_service_zone")
                    .drop("PU_ID")
                    
                    # Add dropoff zone details
                    .join(broadcast(tlc_dim.withColumnRenamed("LocationID","DO_ID")),
                          enriched_df.DOLocationID == F.col("DO_ID"), "left")
                    .withColumnRenamed("Borough","DO_Borough")
                    .withColumnRenamed("Zone","DO_Zone")
                    .withColumnRenamed("service_zone","DO_service_zone") 
                    .drop("DO_ID")
                )
                
                print("✅ Added TLC zone information (boroughs, zones)")
            
            # === CBD CLASSIFICATION ===
            if cbd_dim is not None:
                print("🏢 Adding CBD classification...")
                
                # Get all CBD LocationIDs
                cbd_locations = [row['LocationID'] for row in cbd_dim.select("LocationID").collect()]
                
                enriched_df = (enriched_df
                    # Mark pickup locations in CBD
                    .withColumn("PU_in_cbd", 
                               F.when(F.col("PULocationID").isin(cbd_locations), 1).otherwise(0))
                    
                    # Mark dropoff locations in CBD  
                    .withColumn("DO_in_cbd", 
                               F.when(F.col("DOLocationID").isin(cbd_locations), 1).otherwise(0))
                    
                    # Create trip type classification
                    .withColumn("trip_cbd_type", 
                        F.when((F.col("PU_in_cbd") == 1) & (F.col("DO_in_cbd") == 1), "intra_cbd")
                         .when((F.col("PU_in_cbd") == 1) & (F.col("DO_in_cbd") == 0), "outbound_cbd") 
                         .when((F.col("PU_in_cbd") == 0) & (F.col("DO_in_cbd") == 1), "inbound_cbd")
                         .otherwise("non_cbd")
                    )
                )
                
                print("✅ Added CBD classification (trip_cbd_type)")
            
            # === ADDITIONAL SPATIAL FEATURES ===
            print("🌍 Adding additional spatial features...")
            
            # Cross-borough analysis
            enriched_df = enriched_df.withColumn(
                "is_cross_borough",
                F.when((F.col("PU_Borough").isNotNull()) & (F.col("DO_Borough").isNotNull()),
                       F.col("PU_Borough") != F.col("DO_Borough"))
                .otherwise(False)
            )
            
            # Manhattan involvement
            enriched_df = enriched_df.withColumn(
                "involves_manhattan",
                (F.col("PU_Borough") == "Manhattan") | (F.col("DO_Borough") == "Manhattan")
            )
            
            # Airport involvement (known airport LocationIDs)
            airport_locationids = [1, 132, 138]  # JFK, LaGuardia, Newark
            enriched_df = enriched_df.withColumn(
                "involves_airport",
                F.col("PULocationID").isin(airport_locationids) | 
                F.col("DOLocationID").isin(airport_locationids)
            )
            
            print("✅ Added additional spatial features")
            
            return enriched_df
        
        # === STEP 2: Ensure CBD mapping exists ===
        print("\n📂 Step 2: Ensuring CBD mapping exists...")
        cbd_csv_path = build_cbd_locationids_csv_if_needed_recreated()
        
        if cbd_csv_path is None:
            print("❌ Failed to create CBD mapping")
            return None
        
        # === STEP 3: Load dimension tables ===
        print("\n📋 Step 3: Loading dimension tables...")
        tlc_dim, cbd_dim = load_tlc_and_cbd_dimensions_recreated()
        
        # === STEP 4: Create spatial lookup table ===
        print(f"\n🗺️ Step 4: Creating spatial lookup table...")
        
        # Get all possible LocationID combinations
        all_pickup_locations = tlc_dim.select(F.col("LocationID").alias("PULocationID")).distinct()
        all_dropoff_locations = tlc_dim.select(F.col("LocationID").alias("DOLocationID")).distinct()
        
        # Create PU and DO combinations (Cartesian product)
        spatial_combinations = all_pickup_locations.crossJoin(all_dropoff_locations)
        
        print(f"📊 Creating {spatial_combinations.count():,} spatial feature combinations...")
        
        # Add spatial features
        spatial_features_df = add_trip_classification_labels_recreated(
            spatial_combinations, tlc_dim, cbd_dim
        )
        
        # === STEP 5: Save per month (align with other external datasets) ===
        print(f"\n💾 Step 5: Saving spatial features lookup table...")
        
        # Spatial features are static; write the same lookup table for each month
        years = [2023, 2024]
        months = list(range(1, 13))
        
        saved_files = []
        
        for year in years:
            for month in months:
                save_path = f"./lake/external_features/spatial/year={year}/month={month:02d}"
                
                try:
                    spatial_features_df.write.mode("overwrite").parquet(save_path)
                    
                    saved_files.append({
                        'year': year,
                        'month': month,
                        'combinations': spatial_features_df.count(),
                        'path': save_path
                    })
                    
                    print(f"✅ Saved spatial lookup for {year}-{month:02d}: {save_path}")
                    
                except Exception as e:
                    print(f"❌ Failed to save {year}-{month:02d}: {e}")
        
        # === STEP 6: Create summary information ===
        print(f"\n📋 Step 6: Creating spatial features summary...")
        
        summary = {
            'total_months_saved': len(saved_files),
            'years_covered': years,
            'features_created': actual_spatial_features,
            'feature_count': len(actual_spatial_features),
            'total_combinations': spatial_features_df.count(),
            'monthly_files': saved_files[:6],  # Show first 6
            'feature_descriptions': {
                'PU_in_cbd': 'Pickup location in Central Business District (0/1)',
                'DO_in_cbd': 'Dropoff location in Central Business District (0/1)',
                'trip_cbd_type': 'Trip classification: intra_cbd, inbound_cbd, outbound_cbd, non_cbd',
                'PU_Borough': 'Pickup borough name (Manhattan, Brooklyn, Queens, Bronx, Staten Island)',
                'DO_Borough': 'Dropoff borough name',
                'PU_Zone': 'Pickup taxi zone name (265 zones)',
                'DO_Zone': 'Dropoff taxi zone name',
                'PU_service_zone': 'Pickup service zone type',
                'DO_service_zone': 'Dropoff service zone type',
                'is_cross_borough': 'Trip crosses borough boundaries (True/False)',
                'involves_manhattan': 'Trip involves Manhattan as pickup or dropoff (True/False)',
                'involves_airport': 'Trip involves airport zones (JFK, LaGuardia, Newark)'
            },
            'cbd_zones_count': cbd_dim.count(),
            'total_zones': tlc_dim.count()
        }
        
        # Save summary information
        import json
        summary_path = "./lake/external_features/spatial/spatial_summary.json"
        import os
        os.makedirs(os.path.dirname(summary_path), exist_ok=True)
        
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"✅ Summary saved: {summary_path}")
        
        # === STEP 7: Display results ===
        print(f"\n🎉 SPATIAL FEATURES PROCESSING COMPLETE!")
        print(f"="*60)
        print(f"🗺️ Coverage: All {summary['total_zones']} NYC taxi zones")
        print(f"🏢 CBD zones: {summary['cbd_zones_count']} (14.6%)")
        print(f"📊 Spatial combinations: {summary['total_combinations']:,}")
        print(f"📂 Months saved: {len(saved_files)}")
        print(f"🏷️ Features per lookup: {len(actual_spatial_features)}")
        
        print(f"\n🗺️ SPATIAL FEATURES CREATED:")
        for i, feature in enumerate(actual_spatial_features, 1):
            desc = summary['feature_descriptions'].get(feature, '')
            print(f"  {i:2d}. {feature:<20} - {desc[:50]}{'...' if len(desc) > 50 else ''}")
        
        return {
            'summary': summary,
            'saved_files': saved_files,
            'spatial_lookup': spatial_features_df,
            'tlc_dim': tlc_dim,
            'cbd_dim': cbd_dim
        }
        
    except Exception as e:
        print(f"❌ Spatial processing failed: {e}")
        import traceback
        traceback.print_exc()
        return None

def load_monthly_spatial_features(year, month):
    """
    Load the spatial feature lookup table for a given year and month.
    
    Args:
        year (int): Year
        month (int): Month
    
    Returns:
        DataFrame: Spatial feature lookup table for the specified month
    """
    
    load_path = f"./lake/external_features/spatial/year={year}/month={month:02d}"
    
    try:
        spatial_data = spark.read.parquet(load_path)
        print(f"✅ Loaded spatial lookup for {year}-{month:02d}: {spatial_data.count():,} combinations")
        return spatial_data
    except Exception as e:
        print(f"❌ Failed to load spatial data for {year}-{month:02d}: {e}")
        return None

def test_spatial_data_files():
    """
    测试已保存的spatial数据文件
    """
    
    print("🔍 TESTING SAVED SPATIAL DATA FILES")
    print("="*50)
    
    # 测试几个月份
    test_months = [(2023, 1), (2023, 6), (2024, 1)]
    
    for year, month in test_months:
        try:
            spatial_data = load_monthly_spatial_features(year, month)
            if spatial_data:
                print(f"✅ {year}-{month:02d}: {spatial_data.count():,} combinations, {len(spatial_data.columns)} columns")
                
                # 显示CBD trip distribution sample
                if year == 2023 and month == 1:
                    print(f"\n🏢 CBD trip type distribution sample:")
                    spatial_data.groupBy("trip_cbd_type").count().orderBy(F.desc("count")).show()
                    
                    print(f"\n📊 Sample spatial features:")
                    spatial_data.select([
                        "PULocationID", "DOLocationID", "PU_Borough", "DO_Borough",
                        "trip_cbd_type", "is_cross_borough", "involves_manhattan"
                    ]).limit(5).show(truncate=False)
        except Exception as e:
            print(f"❌ {year}-{month:02d}: {e}")

# ============================================================================
# 可执行代码
# ============================================================================

# 运行spatial数据处理和保存
print("🚀 STARTING SPATIAL FEATURES PROCESSING AND MONTHLY STORAGE")
print("="*70)

spatial_result = create_monthly_spatial_features_and_save()

if spatial_result:
    print("\n🔍 Testing spatial data files...")
    test_spatial_data_files()
    
    print("\n🎉 ALL SPATIAL PROCESSING COMPLETE!")
    print("📂 Spatial features are now available monthly for 2023-2024")
    print("🔗 Ready for integration with trip data via PULocationID + DOLocationID")
else:
    print("\n❌ Spatial processing failed")

print("\n📁 USAGE:")
print("# To load specific month:")
print("# spatial_jan_2023 = load_monthly_spatial_features(2023, 1)")
print("# Then join with trip data on PULocationID and DOLocationID")

### External Data: NOAA nyc weather data


In [ ]:
def process_noaa_weather_data_optimized(quality_results=None):
    """
    Optimized weather data processing based on quality assessment results.
    
    Args:
        quality_results (dict): Results from data quality assessment
    
    Returns:
        DataFrame: Processed and feature-engineered weather data
    """
    
    print("🔧 OPTIMIZED WEATHER DATA PROCESSING")
    print("="*50)
    
    try:
        # Load raw data
        weather_csv = "./lake/landing/lookups/NOAA nyc weather.csv"
        weather_raw = spark.read.csv(weather_csv, header=True)
        
        print(f"📂 Loaded raw data: {weather_raw.count():,} records")
        
        # === STEP 1: STATION SELECTION STRATEGY ===
        print(f"\n1️⃣ STATION SELECTION STRATEGY")
        print("-" * 30)
        
        # Based on quality assessment, prioritize stations with best temperature data
        primary_stations = [
            "USW00014734",  # Newark Liberty International Airport (best temperature data)
            "USW00094728",  # La Guardia Airport
            "USW00014732",  # JFK Airport
            "USC00305816"   # NYC Central Park (if available)
        ]
        
        # Find available stations and their data quality
        available_stations = weather_raw.select("STATION", "NAME").distinct().collect()
        station_names = {row['STATION']: row['NAME'] for row in available_stations}
        
        selected_stations = []
        for station in primary_stations:
            if station in station_names:
                selected_stations.append(station)
                print(f"✅ Using: {station} - {station_names[station]}")
        
        if not selected_stations:
            # Fallback: use top 3 stations by record count
            fallback_stations = (weather_raw.groupBy("STATION", "NAME").count()
                                .orderBy(F.desc("count")).limit(3).collect())
            selected_stations = [row['STATION'] for row in fallback_stations]
            print(f"⚠️ Using fallback stations: {selected_stations}")
        
        # === STEP 2: DATA FILTERING AND CLEANING ===
        print(f"\n2️⃣ DATA FILTERING AND CLEANING")
        print("-" * 30)
        
        # Filter to selected stations
        weather_filtered = weather_raw.filter(F.col("STATION").isin(selected_stations))
        
        # Parse and validate dates
        weather_cleaned = (weather_filtered
            .withColumn("weather_date", F.to_date("DATE", "yyyy-MM-dd"))
            .filter(F.col("weather_date").isNotNull())
            # Filter to 2023-2024 for trip data alignment
            .filter(F.year("weather_date").isin([2023, 2024]))
        )
        
        print(f"📅 After date filtering: {weather_cleaned.count():,} records")
        
        # === STEP 3: TEMPERATURE DATA PROCESSING ===
        print(f"\n3️⃣ TEMPERATURE DATA PROCESSING")
        print("-" * 30)
        
        # Clean and convert temperature data (handle missing, "T" traces, unrealistic values)
        weather_temp_processed = (weather_cleaned
            # TMAX processing
            .withColumn("tmax_raw", 
                       F.when((F.col("TMAX").isNotNull()) & 
                              (F.col("TMAX") != "") & 
                              (F.col("TMAX") != "T"), 
                              F.col("TMAX").cast("double")).otherwise(None))
            .withColumn("tmax_f", 
                       F.when((F.col("tmax_raw") >= -50) & (F.col("tmax_raw") <= 120), 
                              F.col("tmax_raw")).otherwise(None))
            .withColumn("tmax_c", (F.col("tmax_f") - 32) * 5/9)
            
            # TMIN processing
            .withColumn("tmin_raw", 
                       F.when((F.col("TMIN").isNotNull()) & 
                              (F.col("TMIN") != "") & 
                              (F.col("TMIN") != "T"), 
                              F.col("TMIN").cast("double")).otherwise(None))
            .withColumn("tmin_f", 
                       F.when((F.col("tmin_raw") >= -60) & (F.col("tmin_raw") <= 100), 
                              F.col("tmin_raw")).otherwise(None))
            .withColumn("tmin_c", (F.col("tmin_f") - 32) * 5/9)
            
            # TAVG processing (if available)
            .withColumn("tavg_raw", 
                       F.when((F.col("TAVG").isNotNull()) & 
                              (F.col("TAVG") != "") & 
                              (F.col("TAVG") != "T"), 
                              F.col("TAVG").cast("double")).otherwise(None))
            .withColumn("tavg_f", 
                       F.when((F.col("tavg_raw") >= -55) & (F.col("tavg_raw") <= 110), 
                              F.col("tavg_raw")).otherwise(None))
            .withColumn("tavg_c", (F.col("tavg_f") - 32) * 5/9)
        )
        
        # === STEP 4: PRECIPITATION AND WEATHER DATA ===
        print(f"\n4️⃣ PRECIPITATION AND WEATHER DATA PROCESSING")
        print("-" * 30)
        
        weather_precip_processed = (weather_temp_processed
            # Precipitation (inches to mm, handle "T" = trace = 0.01)
            .withColumn("prcp_raw", 
                       F.when(F.col("PRCP") == "T", 0.01)
                        .when((F.col("PRCP").isNotNull()) & (F.col("PRCP") != ""), 
                              F.col("PRCP").cast("double"))
                        .otherwise(0.0))
            .withColumn("precipitation_mm", 
                       F.when(F.col("prcp_raw") >= 0, F.col("prcp_raw") * 25.4)
                        .otherwise(0.0))
            
            # Snow (inches to cm)
            .withColumn("snow_raw", 
                       F.when(F.col("SNOW") == "T", 0.01)
                        .when((F.col("SNOW").isNotNull()) & (F.col("SNOW") != ""), 
                              F.col("SNOW").cast("double"))
                        .otherwise(0.0))
            .withColumn("snow_cm", 
                       F.when(F.col("snow_raw") >= 0, F.col("snow_raw") * 2.54)
                        .otherwise(0.0))
            
            # Snow depth
            .withColumn("snwd_raw", 
                       F.when(F.col("SNWD") == "T", 0.01)
                        .when((F.col("SNWD").isNotNull()) & (F.col("SNWD") != ""), 
                              F.col("SNWD").cast("double"))
                        .otherwise(0.0))
            .withColumn("snow_depth_cm", 
                       F.when(F.col("snwd_raw") >= 0, F.col("snwd_raw") * 2.54)
                        .otherwise(0.0))
            
            # Wind speed (mph to km/h)
            .withColumn("wind_raw", 
                       F.when((F.col("AWND").isNotNull()) & (F.col("AWND") != ""), 
                              F.col("AWND").cast("double")).otherwise(None))
            .withColumn("wind_speed_kmh", 
                       F.when((F.col("wind_raw") >= 0) & (F.col("wind_raw") <= 200), 
                              F.col("wind_raw") * 1.60934).otherwise(None))
        )
        
        # === STEP 5: WEATHER TYPE FLAGS ===
        print(f"\n5️⃣ WEATHER TYPE FLAGS PROCESSING")
        print("-" * 30)
        
        weather_flags = (weather_precip_processed
            # Weather type binary flags
            .withColumn("fog_mist", F.when(F.col("WT01").isNotNull(), 1).otherwise(0))
            .withColumn("heavy_fog", F.when(F.col("WT02").isNotNull(), 1).otherwise(0))
            .withColumn("thunder", F.when(F.col("WT03").isNotNull(), 1).otherwise(0))
            .withColumn("ice_pellets", F.when(F.col("WT04").isNotNull(), 1).otherwise(0))
            .withColumn("hail", F.when(F.col("WT05").isNotNull(), 1).otherwise(0))
            .withColumn("glaze_rime", F.when(F.col("WT06").isNotNull(), 1).otherwise(0))
            .withColumn("smoke_haze", F.when(F.col("WT08").isNotNull(), 1).otherwise(0))
            
            # Rain intensity flags
            .withColumn("is_precipitation", F.when(F.col("precipitation_mm") > 0, 1).otherwise(0))
            .withColumn("is_light_rain", F.when((F.col("precipitation_mm") > 0) & (F.col("precipitation_mm") <= 2.5), 1).otherwise(0))
            .withColumn("is_moderate_rain", F.when((F.col("precipitation_mm") > 2.5) & (F.col("precipitation_mm") <= 10), 1).otherwise(0))
            .withColumn("is_heavy_rain", F.when((F.col("precipitation_mm") > 10) & (F.col("precipitation_mm") <= 25), 1).otherwise(0))
            .withColumn("is_extreme_rain", F.when(F.col("precipitation_mm") > 25, 1).otherwise(0))
            
            # Snow flags
            .withColumn("is_snow", F.when(F.col("snow_cm") > 0, 1).otherwise(0))
            .withColumn("is_light_snow", F.when((F.col("snow_cm") > 0) & (F.col("snow_cm") <= 2.5), 1).otherwise(0))
            .withColumn("is_heavy_snow", F.when(F.col("snow_cm") > 2.5, 1).otherwise(0))
        )
        
        # === STEP 6: EXTREME TEMPERATURE FLAGS ===
        print(f"\n6️⃣ EXTREME TEMPERATURE FLAGS")
        print("-" * 30)
        
        weather_extremes = (weather_flags
            .withColumn("is_freezing", F.when(F.col("tmin_c") <= 0, 1).otherwise(0))
            .withColumn("is_cold", F.when((F.col("tmax_c") > 0) & (F.col("tmax_c") <= 5), 1).otherwise(0))
            .withColumn("is_cool", F.when((F.col("tmax_c") > 5) & (F.col("tmax_c") <= 15), 1).otherwise(0))
            .withColumn("is_mild", F.when((F.col("tmax_c") > 15) & (F.col("tmax_c") <= 20), 1).otherwise(0))
            .withColumn("is_warm", F.when((F.col("tmax_c") > 20) & (F.col("tmax_c") <= 25), 1).otherwise(0))
            .withColumn("is_hot", F.when(F.col("tmax_c") >= 30, 1).otherwise(0))
            .withColumn("is_very_cold", F.when((F.col("tmax_c") > -5) & (F.col("tmax_c") <= 0), 1).otherwise(0))
            .withColumn("is_extreme_heat", F.when(F.col("tmax_c") >= 35, 1).otherwise(0))
        )
        
        # === STEP 7: CONSOLIDATED FEATURES AND CATEGORIZATION ===
        print(f"\n7️⃣ CONSOLIDATED FEATURES AND CATEGORIZATION")
        print("-" * 30)
        
        weather_consolidated = (weather_extremes
            # Average temperature (prefer tavg_c if available, else avg of min/max)
            .withColumn("temp_avg_calculated", 
                       F.when(F.col("tavg_c").isNotNull(), F.col("tavg_c"))
                        .otherwise((F.col("tmax_c") + F.col("tmin_c")) / 2.0))
            # Temperature range
            .withColumn("temp_range_c", F.col("tmax_c") - F.col("tmin_c"))
            # Fog consolidated flag
            .withColumn("fog_flag", 
                       (F.col("fog_mist").cast("int") + F.col("heavy_fog").cast("int") > 0).cast("int"))
        )
        
        # Weather comfort index (1-5 scale) based on temperature and precipitation
        weather_enhanced = (weather_consolidated
            .withColumn("weather_comfort_index", 
                       F.when((F.col("temp_avg_calculated") >= 18) & (F.col("temp_avg_calculated") <= 24) & (F.col("precipitation_mm") == 0), 5)
                        .when((F.col("temp_avg_calculated") >= 15) & (F.col("temp_avg_calculated") <= 27) & (F.col("precipitation_mm") <= 1), 4)
                        .when((F.col("temp_avg_calculated") >= 10) & (F.col("temp_avg_calculated") <= 30) & (F.col("precipitation_mm") <= 5), 3)
                        .when((F.col("temp_avg_calculated") >= 0) & (F.col("temp_avg_calculated") <= 35) & (F.col("precipitation_mm") <= 20), 2)
                        .otherwise(1))
        )
        
        # === STEP 8: SEASON ===
        print(f"\n8️⃣ SEASON CATEGORIZATION")
        print("-" * 30)
        
        weather_final = (weather_enhanced
            .withColumn("month", F.month("weather_date"))
            .withColumn("year", F.year("weather_date"))
            .withColumn("season", 
                       F.when(F.col("month").isin(12, 1, 2), "winter")
                        .when(F.col("month").isin(3, 4, 5), "spring")
                        .when(F.col("month").isin(6, 7, 8), "summer")
                        .otherwise("fall"))
            .select(
                "weather_date", "year", "month", "season",
                "tmax_c", "tmin_c", "tavg_c",
                "temp_avg_calculated", "temp_range_c",
                "precipitation_mm", "snow_cm", "snow_depth_cm", "wind_speed_kmh",
                "is_precipitation", "is_light_rain", "is_moderate_rain", "is_heavy_rain", "is_extreme_rain",
                "is_snow", "is_light_snow", "is_heavy_snow",
                "is_freezing", "is_cold", "is_cool", "is_mild", "is_warm", "is_hot", "is_very_cold", "is_extreme_heat",
                "fog_mist", "heavy_fog", "fog_flag",
                "weather_comfort_index"
            )
            .orderBy("weather_date")
        )
        
        print(f"\n✅ Processing complete: {weather_final.count():,} daily records")
        return weather_final
    
    except Exception as e:
        print(f"❌ Weather processing failed: {e}")
        import traceback
        print(traceback.format_exc())
        return None

In [4]:
# ========== IMPORT OPTIMIZED WEATHER PROCESSING FUNCTIONS ==========
from preprocessing.noaa_weather_data import (
    assess_noaa_weather_data_quality,
    process_noaa_weather_data_optimized_v2,
    save_optimized_weather_to_external_features
)

print("✅ Weather processing functions imported successfully!")

✅ Weather processing functions imported successfully!


In [ ]:
from pyspark.sql import functions as F

# Read only partitioned parquet files; ignore weather_summary.json
existing_weather = (
    spark.read
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.parquet")
    .parquet("./lake/external_features/weather")
)

# Minimal set of features to keep
keep = [
    "weather_date", "year", "month", "season",
    "temp_avg_calculated", "is_freezing", "is_hot",
    "precipitation_mm", "is_snow", "wind_speed_kmh",
    "weather_comfort_index", "fog_flag"
]

# If fog_flag missing, derive from fog_mist/heavy_fog flags
available = set(existing_weather.columns)
df = existing_weather
if "fog_flag" not in available and ({"fog_mist","heavy_fog"} & available):
    fog_mist_bool = (F.col("fog_mist").cast("int") > 0) if "fog_mist" in available else F.lit(False)
    heavy_fog_bool = (F.col("heavy_fog").cast("int") > 0) if "heavy_fog" in available else F.lit(False)
    df = df.withColumn("fog_flag", (F.coalesce(fog_mist_bool, F.lit(False)) | F.coalesce(heavy_fog_bool, F.lit(False))).cast("int"))
# Select available columns and write optimized subset
final_cols = [c for c in keep if c in df.columns]
optimized = df.select(*final_cols)

out_path = "./lake/external_features/weather_optimized"
optimized.write.mode("overwrite").partitionBy("year","month").parquet(out_path)

print("✅ Saved to:", out_path)
print("rows:", optimized.count(), "cols:", len(optimized.columns))

✅ Saved to: ./lake/external_features/weather_optimized
rows: 731 cols: 12


In [ ]:
# check weather_optimized schema
path = "./lake/external_features/weather_optimized"

optimized = (spark.read
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.parquet")
    .parquet(path)
)

print("rows:", optimized.count(), "cols:", len(optimized.columns))
optimized.printSchema()


rows: 731 cols: 10
root
 |-- weather_date: date (nullable = true)
 |-- season: string (nullable = true)
 |-- temp_avg_calculated: double (nullable = true)
 |-- is_freezing: integer (nullable = true)
 |-- is_hot: integer (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- is_snow: integer (nullable = true)
 |-- wind_speed_kmh: double (nullable = true)
 |-- weather_comfort_index: double (nullable = true)
 |-- fog_flag: integer (nullable = true)



### External Dataset: Holiday data + time-series features

In [ ]:
def test_temporal_features_with_visualization(year=2023, month=1, sample_fraction=0.01):
    """
    Complete test of temporal features with improved visualization.
    
    Args:
        year (int): Year to test (default: 2023)
        month (int): Month to test (default: 1)
        sample_fraction (float): Sample fraction for testing (default: 0.01)
    
    Returns:
        tuple: (temporal_data, visualization_stats)
    """
    
    print("🎨 COMPLETE TEMPORAL FEATURES TEST WITH IMPROVED VISUALIZATION")
    print("="*70)
    
    # === STEP 1: LOAD HOLIDAYS DIMENSION ===
    print("📅 Step 1: Loading NYC federal holidays...")
    try:
        holidays_dim = load_nyc_holidays_dimension()
        if holidays_dim is not None:
            print("✅ Holiday dimension loaded successfully")
        else:
            print("⚠️ Holiday dimension failed to load")
    except Exception as e:
        print(f"❌ Holiday loading error: {e}")
        holidays_dim = None
    
    # === STEP 2: LOAD SAMPLE UBER DATA ===
    print(f"\n📂 Step 2: Loading sample Uber data ({year}-{month:02d})...")
    try:
        sample_data = extract_uber_data_for_analysis(
            year_list=[year],
            month_list=[month],
            sample_fraction=sample_fraction
        )
        
        if sample_data is not None:
            sample_count = sample_data.count()
            print(f"✅ Loaded {sample_count:,} sample trips")
            
            if sample_count == 0:
                print("❌ No trips found in sample")
                return None, None
        else:
            print("❌ Failed to load sample data")
            return None, None
            
    except Exception as e:
        print(f"❌ Data loading error: {e}")
        return None, None
    
    # === STEP 3: ADD TEMPORAL FEATURES ===
    print(f"\n🕐 Step 3: Adding temporal features...")
    try:
        temporal_data = add_temporal_features(sample_data, holidays_dim)
        print("✅ Temporal features added successfully")
        
        # Show added features
        original_cols = set(sample_data.columns)
        new_cols = [col for col in temporal_data.columns if col not in original_cols]
        print(f"📋 New temporal features added ({len(new_cols)}):")
        for col in new_cols:
            print(f"  • {col}")
            
    except Exception as e:
        print(f"❌ Temporal feature addition failed: {e}")
        return None, None
    
    # === STEP 4: BASIC TEMPORAL ANALYSIS ===
    print(f"\n📈 Step 4: Running basic temporal analysis...")
    try:
        analysis_results = analyze_temporal_patterns(temporal_data)
        print("✅ Basic analysis completed")
    except Exception as e:
        print(f"⚠️ Basic analysis failed: {e}")
        analysis_results = {}
    
    # === STEP 5: CREATE IMPROVED VISUALIZATION ===
    print(f"\n🎨 Step 5: Creating improved temporal visualization...")
    
    try:
        viz_stats = visualize_temporal_patterns_improved(
            temporal_data,
            sample_size=min(50000, temporal_data.count()),
            save_plots=True
        )
        
        if viz_stats is not None:
            print("✅ Improved visualization created successfully")
        else:
            print("⚠️ Visualization completed with warnings")
            
    except Exception as e:
        print(f"❌ Visualization failed: {e}")
        print("Trying basic visualization...")
        
        try:
            viz_stats = create_basic_temporal_plots(temporal_data)
            print("✅ Basic visualization completed")
        except Exception as e2:
            print(f"❌ All visualization attempts failed: {e2}")
            viz_stats = None
    
    # === STEP 6: DETAILED INSIGHTS ===
    print(f"\n🔍 Step 6: Generating detailed insights...")
    
    try:
        # Get sample for detailed analysis
        insight_sample = temporal_data.limit(20000).toPandas()
        
        insights = generate_temporal_insights(insight_sample)
        
        print(f"\n📊 TEMPORAL INSIGHTS:")
        for insight in insights:
            print(f"  💡 {insight}")
            
    except Exception as e:
        print(f"⚠️ Insights generation failed: {e}")
        insights = []
    
    # === STEP 7: SUMMARY REPORT ===
    print(f"\n📋 Step 7: Final summary report...")
    
    try:
        final_count = temporal_data.count()
        total_cols = len(temporal_data.columns)
        
        print(f"\n" + "="*50)
        print(f"🎉 TEMPORAL FEATURES TEST COMPLETE!")
        print(f"="*50)
        print(f"📊 Dataset: {final_count:,} trips, {total_cols} columns")
        print(f"📅 Period: {year}-{month:02d}")
        print(f"🔢 Sample rate: {sample_fraction*100}%")
        
        if analysis_results:
            print(f"\n📈 KEY METRICS:")
            print(f"  🕐 Peak hour: {analysis_results.get('peak_hours', {}).index[0] if 'peak_hours' in analysis_results else 'N/A'}:00")
            print(f"  📅 Weekend trips: {analysis_results.get('weekend_pct', 0):.1f}%")
            print(f"  🎉 Holiday trips: {analysis_results.get('holiday_pct', 0):.1f}%")
            print(f"  🚗 Rush hour trips: {analysis_results.get('rush_hour_pct', 0):.1f}%")
        
        print(f"\n📁 FILES CREATED:")
        print(f"  📊 temporal_patterns_analysis_improved.png")
        
        print(f"\n🎯 NEXT STEPS:")
        print(f"  ✅ Temporal features are working correctly")
        print(f"  📊 Visualization dashboard created")
        print(f"  🔄 Ready to combine with spatial features")
        print(f"  📈 Ready for profitability analysis")
        
        return temporal_data, {
            'analysis_results': analysis_results,
            'viz_stats': viz_stats,
            'insights': insights,
            'final_count': final_count,
            'total_columns': total_cols
        }
        
    except Exception as e:
        print(f"❌ Summary generation failed: {e}")
        return temporal_data, {'error': str(e)}

def visualize_temporal_patterns_improved(df, sample_size=50000, save_plots=True):
    """
    Improved temporal visualization with fixed label overlapping issues.
    """
    
    print("📊 CREATING IMPROVED TEMPORAL PATTERNS VISUALIZATION")
    print("="*50)
    
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        import pandas as pd
        import numpy as np
        
        # Enhanced styling
        plt.style.use('default')
        sns.set_palette("husl")
        plt.rcParams.update({
            'font.size': 10,
            'axes.titlesize': 12,
            'axes.labelsize': 11,
            'xtick.labelsize': 9,
            'ytick.labelsize': 9,
            'legend.fontsize': 9
        })
        
        # Sample data
        total_count = df.count()
        actual_sample_size = min(sample_size, total_count)
        viz_df = df.limit(actual_sample_size).toPandas()
        
        print(f"📈 Creating visualization from {actual_sample_size:,} trips...")
        
        # Create figure with optimal spacing
        fig = plt.figure(figsize=(24, 20))
        
        # === 1. HOURLY DISTRIBUTION ===
        plt.subplot(4, 3, 1)
        hourly_counts = viz_df['pickup_hour'].value_counts().sort_index()
        bars = plt.bar(hourly_counts.index, hourly_counts.values, 
                      color='steelblue', alpha=0.7, edgecolor='navy')
        
        # Highlight rush hours
        for i, bar in enumerate(bars):
            if i in [7, 8, 9]:  # Morning rush
                bar.set_color('red')
                bar.set_alpha(0.8)
            elif i in [17, 18, 19]:  # Evening rush
                bar.set_color('orange')
                bar.set_alpha(0.8)
        
        plt.xlabel('Hour of Day')
        plt.ylabel('Number of Trips')
        plt.title('Hourly Trip Distribution\n(Red: Morning Rush, Orange: Evening Rush)')
        plt.xticks(range(0, 24, 2))
        plt.grid(True, alpha=0.3)
        
        # === 2. DAY OF WEEK PATTERN ===
        plt.subplot(4, 3, 2)
        day_names = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
        dow_counts = viz_df['pickup_day_of_week'].value_counts().sort_index()
        
        colors = ['red' if i in [1, 7] else 'steelblue' for i in dow_counts.index]
        plt.bar(range(len(dow_counts)), dow_counts.values, color=colors, alpha=0.7)
        
        plt.xlabel('Day of Week')
        plt.ylabel('Number of Trips')
        plt.title('Weekly Pattern (Red: Weekends)')
        plt.xticks(range(len(day_names)), day_names)
        plt.grid(True, alpha=0.3)
        
        # === 3. RUSH HOUR BREAKDOWN ===
        plt.subplot(4, 3, 3)
        rush_counts = viz_df['rush_hour_label'].value_counts()
        colors = plt.cm.Set3(np.linspace(0, 1, len(rush_counts)))
        
        plt.pie(rush_counts.values, labels=rush_counts.index, autopct='%1.1f%%', 
               colors=colors, startangle=90)
        plt.title('Rush Hour Distribution')
        
        # === 4. WEEKDAY VS WEEKEND ===
        plt.subplot(4, 3, 4)
        weekend_data = viz_df.groupby(['is_weekend', 'pickup_hour']).size().unstack(0, fill_value=0)
        
        x = range(24)
        width = 0.35
        plt.bar([i - width/2 for i in x], weekend_data[0], width, 
               label='Weekday', alpha=0.7, color='steelblue')
        plt.bar([i + width/2 for i in x], weekend_data[1], width, 
               label='Weekend', alpha=0.7, color='red')
        
        plt.xlabel('Hour of Day')
        plt.ylabel('Average Trips')
        plt.title('Weekday vs Weekend Hourly Patterns')
        plt.legend()
        plt.xticks(range(0, 24, 3))
        plt.grid(True, alpha=0.3)
        
        # === 5. HOLIDAY IMPACT ===
        plt.subplot(4, 3, 5)
        holiday_comparison = viz_df.groupby(['is_holiday', 'pickup_hour']).size().unstack(0, fill_value=0)
        
        plt.plot(holiday_comparison.index, holiday_comparison[0], 
                label='Regular Days', linewidth=2, color='steelblue')
        
        if 1 in holiday_comparison.columns:
            plt.plot(holiday_comparison.index, holiday_comparison[1], 
                    label='Holidays', linewidth=2, color='red', linestyle='--')
        
        plt.xlabel('Hour of Day')
        plt.ylabel('Average Trips')
        plt.title('Holiday vs Regular Day Patterns')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # === 6. TIME PERIOD SUMMARY ===
        plt.subplot(4, 3, 6)
        period_counts = viz_df['time_period'].value_counts()
        colors = ['gold', 'lightcoral', 'lightblue', 'mediumpurple']
        
        plt.bar(period_counts.index, period_counts.values, color=colors, alpha=0.8)
        plt.xlabel('Time Period')
        plt.ylabel('Number of Trips')
        plt.title('Time Period Distribution')
        plt.xticks(rotation=0)  # No rotation needed for short labels
        plt.grid(True, alpha=0.3)
        
        # === 7. TRIP DENSITY HEATMAP ===
        plt.subplot(4, 3, 7)
        heatmap_data = viz_df.groupby(['pickup_day_of_week', 'pickup_hour']).size().unstack(fill_value=0)
        
        sns.heatmap(heatmap_data, cmap='YlOrRd', cbar_kws={'label': 'Number of Trips'})
        plt.xlabel('Hour of Day')
        plt.ylabel('Day of Week')
        plt.title('Trip Density Heatmap')
        plt.yticks(range(len(day_names)), day_names, rotation=0)
        
        # === 8. MONTHLY/DAILY PATTERN ===
        plt.subplot(4, 3, 8)
        if viz_df['pickup_month'].nunique() > 1:
            monthly_counts = viz_df['pickup_month'].value_counts().sort_index()
            month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
            
            plt.bar(range(len(monthly_counts)), monthly_counts.values, 
                   color='lightgreen', alpha=0.7)
            plt.xlabel('Month')
            plt.ylabel('Number of Trips')
            plt.title('Monthly Distribution')
            plt.xticks(range(len(monthly_counts)), 
                      [month_names[i-1] for i in monthly_counts.index])
        else:
            viz_df['pickup_day'] = pd.to_datetime(viz_df['pickup_datetime']).dt.day
            daily_counts = viz_df['pickup_day'].value_counts().sort_index()
            plt.plot(daily_counts.index, daily_counts.values, marker='o', linewidth=2)
            plt.xlabel('Day of Month')
            plt.ylabel('Number of Trips')
            plt.title('Daily Pattern (Within Month)')
            plt.grid(True, alpha=0.3)
        
        # === 9. WEEKEND PRIME TIME ===
        plt.subplot(4, 3, 9)
        weekend_df = viz_df[viz_df['is_weekend'] == 1]
        
        if len(weekend_df) > 0:
            weekend_hourly = weekend_df['pickup_hour'].value_counts().sort_index()
            bars = plt.bar(weekend_hourly.index, weekend_hourly.values, 
                          color='purple', alpha=0.7)
            
            # Highlight weekend prime time
            prime_hours = list(range(20, 24)) + list(range(0, 3))
            for i, bar in enumerate(bars):
                if weekend_hourly.index[i] in prime_hours:
                    bar.set_color('gold')
                    bar.set_alpha(0.9)
            
            plt.xlabel('Hour of Day')
            plt.ylabel('Weekend Trips')
            plt.title('Weekend Hourly Pattern\n(Gold: Prime Time 8PM-2AM)')
            plt.xticks(range(0, 24, 3))
        else:
            plt.text(0.5, 0.5, 'No Weekend Data\nin Sample', 
                    ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('Weekend Analysis')
        
        plt.grid(True, alpha=0.3)
        
        # === 10. IMPROVED TRIP CHARACTERISTICS ===
        plt.subplot(4, 3, 10)
        
        rush_summary = viz_df.groupby('rush_hour_label').agg({
            'trip_miles': 'mean',
            'trip_time': 'mean'
        }).round(2)
        
        if len(rush_summary) > 0:
            # Better label mapping
            label_map = {
                'morning_rush': 'Morning\nRush',
                'evening_rush': 'Evening\nRush', 
                'business_hours': 'Business\nHours',
                'off_peak': 'Off\nPeak',
                'weekend_day': 'Weekend\nDay',
                'weekend_prime': 'Weekend\nPrime',
                'late_night': 'Late\nNight',
                'holiday_period': 'Holiday'
            }
            
            # Limit to top categories to avoid crowding
            top_categories = viz_df['rush_hour_label'].value_counts().head(6)
            filtered_summary = rush_summary.loc[top_categories.index]
            
            x_pos = range(len(filtered_summary))
            
            # Create twin axis
            ax1 = plt.gca()
            ax2 = ax1.twinx()
            
            # Plot with better spacing
            width = 0.35
            bars1 = ax1.bar([x - width/2 for x in x_pos], filtered_summary['trip_miles'], 
                           width=width, color='steelblue', alpha=0.8, label='Avg Miles')
            bars2 = ax2.bar([x + width/2 for x in x_pos], filtered_summary['trip_time']/60, 
                           width=width, color='red', alpha=0.8, label='Avg Minutes')
            
            # FIXED LABELS - No overlapping
            clean_labels = [label_map.get(label, label.replace('_', '\n')) 
                           for label in filtered_summary.index]
            
            ax1.set_xticks(x_pos)
            ax1.set_xticklabels(clean_labels, fontsize=9, ha='center')
            
            # Styling
            ax1.set_ylabel('Average Miles', color='steelblue', fontweight='bold')
            ax2.set_ylabel('Average Minutes', color='red', fontweight='bold')
            ax1.tick_params(axis='y', labelcolor='steelblue')
            ax2.tick_params(axis='y', labelcolor='red')
            
            plt.title('Trip Characteristics by Time Period\n(FIXED: No Label Overlap)', fontweight='bold')
            
            # Add values on bars
            for bar1, bar2 in zip(bars1, bars2):
                h1 = bar1.get_height()
                h2 = bar2.get_height()
                ax1.text(bar1.get_x() + bar1.get_width()/2., h1 + 0.05,
                        f'{h1:.1f}', ha='center', va='bottom', 
                        color='steelblue', fontweight='bold', fontsize=8)
                ax2.text(bar2.get_x() + bar2.get_width()/2., h2 + 0.2,
                        f'{h2:.1f}', ha='center', va='bottom',
                        color='red', fontweight='bold', fontsize=8)
            
            # Legend
            lines1, labels1 = ax1.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
        
        # === 11. CORRELATION HEATMAP ===
        plt.subplot(4, 3, 11)
        temporal_cols = ['pickup_hour', 'pickup_day_of_week', 'is_weekend', 
                        'is_holiday', 'is_rush_hour']
        available_cols = [col for col in temporal_cols if col in viz_df.columns]
        
        if len(available_cols) >= 2:
            corr_matrix = viz_df[available_cols].corr()
            sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
                       square=True, cbar_kws={'label': 'Correlation'})
            plt.title('Temporal Features Correlation')
            plt.xticks(rotation=45)
            plt.yticks(rotation=0)
        
        # === 12. SUMMARY STATISTICS ===
        plt.subplot(4, 3, 12)
        
        peak_hour = viz_df['pickup_hour'].mode().iloc[0] if len(viz_df) > 0 else 0
        rush_pct = (viz_df['is_rush_hour']==1).mean()*100 if 'is_rush_hour' in viz_df.columns else 0
        weekend_pct = (viz_df['is_weekend']==1).mean()*100 if 'is_weekend' in viz_df.columns else 0
        holiday_pct = (viz_df['is_holiday']==1).mean()*100 if 'is_holiday' in viz_df.columns else 0
        
        stats_text = f"""TEMPORAL ANALYSIS SUMMARY
        
Sample Size: {len(viz_df):,} trips

🕐 PEAK PATTERNS:
Busiest Hour: {peak_hour:02d}:00
Rush Hour Trips: {rush_pct:.1f}%

📅 CALENDAR PATTERNS:
Weekend Trips: {weekend_pct:.1f}%
Holiday Trips: {holiday_pct:.1f}%

🚗 RUSH HOUR BREAKDOWN:"""
        
        if 'rush_hour_label' in viz_df.columns:
            for label, count in viz_df['rush_hour_label'].value_counts().head(4).items():
                pct = count / len(viz_df) * 100
                stats_text += f"\n{label.replace('_', ' ').title()}: {pct:.1f}%"
        
        plt.text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, 
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8))
        plt.axis('off')
        plt.title('Summary Statistics', pad=20, fontweight='bold')
        
        # Enhanced layout
        plt.tight_layout(pad=4.0, h_pad=4.0, w_pad=3.0)
        
        # Save with higher quality
        if save_plots:
            output_file = "temporal_patterns_analysis_improved.png"
            plt.savefig(output_file, dpi=300, bbox_inches='tight', 
                       facecolor='white', edgecolor='none')
            print(f"💾 Saved improved visualization: {output_file}")
        
        plt.show()
        
        # Return summary statistics
        return {
            'sample_size': len(viz_df),
            'peak_hour': peak_hour,
            'rush_hour_pct': rush_pct,
            'weekend_pct': weekend_pct,
            'holiday_pct': holiday_pct,
            'rush_breakdown': viz_df['rush_hour_label'].value_counts().to_dict() if 'rush_hour_label' in viz_df.columns else {}
        }
        
    except Exception as e:
        print(f"❌ Improved visualization failed: {e}")
        import traceback
        print(f"Traceback: {traceback.format_exc()}")
        return None

def generate_temporal_insights(df):
    """
    Generate business insights from temporal patterns.
    """
    insights = []
    
    try:
        # Peak hour analysis
        peak_hour = df['pickup_hour'].mode().iloc[0]
        insights.append(f"Peak demand occurs at {peak_hour:02d}:00")
        
        # Weekend vs weekday
        if 'is_weekend' in df.columns:
            weekend_avg = df[df['is_weekend']==1].groupby('pickup_hour').size().mean()
            weekday_avg = df[df['is_weekend']==0].groupby('pickup_hour').size().mean()
            
            if weekend_avg > weekday_avg:
                insights.append("Weekend hours show higher average demand than weekdays")
            else:
                insights.append("Weekday hours show higher average demand than weekends")
        
        # Rush hour efficiency
        if 'rush_hour_label' in df.columns and 'trip_miles' in df.columns:
            rush_miles = df[df['rush_hour_label'].isin(['morning_rush', 'evening_rush'])]['trip_miles'].mean()
            off_peak_miles = df[df['rush_hour_label']=='off_peak']['trip_miles'].mean()
            
            if rush_miles > off_peak_miles:
                insights.append("Rush hour trips are longer on average than off-peak trips")
            else:
                insights.append("Off-peak trips are longer on average than rush hour trips")
        
        # Holiday impact
        if 'is_holiday' in df.columns:
            holiday_trips = (df['is_holiday']==1).sum()
            if holiday_trips > 0:
                insights.append(f"Holiday traffic represents {holiday_trips/len(df)*100:.1f}% of total trips")
            else:
                insights.append("No holiday trips detected in sample period")
        
        # Late night patterns
        if 'pickup_hour' in df.columns:
            late_night_trips = df[df['pickup_hour'].isin([22, 23, 0, 1, 2, 3])].shape[0]
            late_night_pct = late_night_trips / len(df) * 100
            insights.append(f"Late night trips (10PM-4AM) represent {late_night_pct:.1f}% of demand")
        
    except Exception as e:
        insights.append(f"Insight generation encountered error: {str(e)}")
    
    return insights

def create_basic_temporal_plots(df):
    """
    Fallback basic visualization if the main one fails.
    """
    try:
        import matplotlib.pyplot as plt
        
        viz_df = df.limit(10000).toPandas()
        
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
        
        # Hourly pattern
        hourly = viz_df['pickup_hour'].value_counts().sort_index()
        ax1.bar(hourly.index, hourly.values)
        ax1.set_title('Hourly Distribution')
        ax1.set_xlabel('Hour')
        ax1.set_ylabel('Trips')
        
        # Day of week
        dow = viz_df['pickup_day_of_week'].value_counts().sort_index()
        ax2.bar(dow.index, dow.values)
        ax2.set_title('Day of Week')
        ax2.set_xlabel('Day (1=Sun, 7=Sat)')
        ax2.set_ylabel('Trips')
        
        # Weekend vs weekday
        weekend = viz_df['is_weekend'].value_counts()
        ax3.pie(weekend.values, labels=['Weekday', 'Weekend'], autopct='%1.1f%%')
        ax3.set_title('Weekend vs Weekday')
        
        # Rush hour
        if 'rush_hour_label' in viz_df.columns:
            rush = viz_df['rush_hour_label'].value_counts()
            ax4.pie(rush.values, labels=rush.index, autopct='%1.1f%%')
            ax4.set_title('Rush Hour Distribution')
        
        plt.tight_layout()
        plt.savefig('basic_temporal_analysis.png', dpi=200, bbox_inches='tight')
        plt.show()
        
        return {'basic_plots': 'created'}
        
    except Exception as e:
        print(f"Basic visualization also failed: {e}")
        return None

# ============================================================================
# RUN THE COMPLETE IMPROVED TEST
# ============================================================================

print("🚀 RUNNING COMPLETE IMPROVED TEMPORAL FEATURES TEST...")
print("This version fixes label overlapping and provides enhanced insights")
print("-" * 70)

# Run the improved test
temporal_result, test_results = test_temporal_features_with_visualization(
    year=2023,
    month=1,
    sample_fraction=0.01  # 1% sample for good visualization quality
)

if temporal_result is not None and test_results is not None:
    print(f"\n🎉 🎉 COMPLETE SUCCESS! 🎉 🎉")
    print(f"✅ All temporal features working perfectly")
    print(f"✅ Improved visualization with fixed label overlapping")
    print(f"✅ Comprehensive insights generated")
    print(f"✅ Ready for integration with spatial features")
    
    if 'analysis_results' in test_results:
        results = test_results['analysis_results']
        print(f"\n📊 FINAL METRICS:")
        print(f"  🎯 Dataset: {test_results.get('final_count', 0):,} trips")
        print(f"  📊 Features: {test_results.get('total_columns', 0)} columns")
        if results:
            print(f"  🕐 Peak hour: {list(results.get('peak_hours', {}).keys())[0] if results.get('peak_hours') else 'N/A'}:00")
            print(f"  📅 Weekend: {results.get('weekend_pct', 0):.1f}%")
            print(f"  🚗 Rush hour: {results.get('rush_hour_pct', 0):.1f}%")
    
else:
    print("❌ Test failed - check error messages above")

print(f"\n📁 Generated files:")
print(f"  📊 temporal_patterns_analysis_improved.png - FIXED visualization dashboard")
print(f"  📈 All label overlapping issues resolved!")

In [ ]:
def create_monthly_temporal_features_and_save():
    """
    Create monthly temporal features based on the Holiday + time-series section and save outputs.
    Reproduce the temporal feature set defined in the External Dataset: Holiday.
    """
    
    print("📅 CREATING AND SAVING MONTHLY TEMPORAL FEATURES")
    print("Based on actual Holiday + time-series processing code")
    print("="*70)
    
    # === Temporal features to create (from outputs) ===
    actual_temporal_features = [
        # Basic time extraction
        'pickup_date',           # 1: Date
        'pickup_hour',           # 2: Hour (0-23)
        'pickup_day_of_week',    # 3: Day of week (0-6, 0=Sunday)
        'pickup_month',          # 4: Month
        'pickup_year',           # 5: Year
        
        # Weekend/weekday
        'is_weekend',            # 6: True/False
        
        # Holiday-related
        'holiday_name',          # 7: Holiday name
        'is_holiday',            # 8: True/False
        
        # Rush hour classification
        'rush_hour_label',       # 9: morning_rush, evening_rush, business_hours, off_peak, late_night, weekend
        'is_rush_hour',          # 10: True/False (aggregate peak hours)
        'time_period'            # 11: morning, afternoon, evening, night
    ]
    
    print(f"📋 Will create {len(actual_temporal_features)} temporal features")
    
    try:
        # === STEP 1: Recreate holiday and temporal processing functions ===
        print("\n📅 Step 1: Loading holidays and creating temporal functions...")
        
        def load_nyc_holidays_dimension_recreated():
            """Recreate function to load the holidays dimension"""
            
            holidays_csv = "./lake/landing/lookups/nyc_federal_holidays_2023_2024.csv"
            
            holidays_df = (spark.read.csv(holidays_csv, header=True)
                          .select(F.to_date("date", "yyyy-MM-dd").alias("holiday_date"),
                                 "holiday_name")
                          .dropna(subset=["holiday_date"])
                          .dropDuplicates(["holiday_date"]))
            
            print(f"✅ Loaded {holidays_df.count()} federal holidays")
            holidays_df.groupBy(F.year("holiday_date").alias("year")).count().show()
            
            return holidays_df
        
        def add_temporal_features_recreated(df, holidays_dim):
            """Recreate function that adds temporal features to a DataFrame"""
            
            print("🕐 Adding comprehensive temporal features...")
            
            # Basic time extraction
            enhanced_df = (df
                .withColumn("pickup_date", F.to_date("pickup_datetime"))
                .withColumn("pickup_hour", F.hour("pickup_datetime"))
                .withColumn("pickup_day_of_week", F.dayofweek("pickup_datetime") - 1)  # 0=Sunday
                .withColumn("pickup_month", F.month("pickup_datetime"))
                .withColumn("pickup_year", F.year("pickup_datetime"))
            )
            
            # Weekend flag
            enhanced_df = enhanced_df.withColumn(
                "is_weekend", 
                F.when(F.col("pickup_day_of_week").isin([0, 6]), True).otherwise(False)  # 0=Sunday, 6=Saturday
            )
            
            # Holiday information
            print("📅 Adding holiday information...")
            enhanced_df = (enhanced_df
                .join(broadcast(holidays_dim), 
                      enhanced_df.pickup_date == holidays_dim.holiday_date, "left")
                .withColumn("is_holiday", F.when(F.col("holiday_name").isNotNull(), True).otherwise(False))
                .drop("holiday_date")
            )
            print("✅ Holiday information added")
            
            # Rush hour classification (based on the derived categories)
            print("🚗 Adding rush hour classifications...")
            enhanced_df = (enhanced_df
                .withColumn("rush_hour_label",
                    F.when((F.col("pickup_hour").between(7, 9)) & (F.col("is_weekend") == False), "morning_rush")
                     .when((F.col("pickup_hour").between(17, 19)) & (F.col("is_weekend") == False), "evening_rush")
                     .when((F.col("pickup_hour").between(9, 17)) & (F.col("is_weekend") == False), "business_hours")
                     .when((F.col("pickup_hour").between(22, 23)) | (F.col("pickup_hour").between(0, 5)), "late_night")
                     .when(F.col("is_weekend") == True, "weekend")
                     .otherwise("off_peak")
                )
                .withColumn("is_rush_hour",
                    F.when(F.col("rush_hour_label").isin(["morning_rush", "evening_rush"]), True).otherwise(False)
                )
                .withColumn("time_period",
                    F.when(F.col("pickup_hour").between(5, 11), "morning")
                     .when(F.col("pickup_hour").between(12, 17), "afternoon")
                     .when(F.col("pickup_hour").between(18, 21), "evening")
                     .otherwise("night")
                )
            )
            
            print("✅ Temporal features added successfully")
            
            new_features = ['pickup_date', 'pickup_hour', 'pickup_day_of_week', 'pickup_month', 'pickup_year',
                           'is_weekend', 'holiday_name', 'is_holiday', 'rush_hour_label', 'is_rush_hour', 'time_period']
            print(f"📊 New temporal features: {new_features}")
            
            return enhanced_df
        
        # === STEP 2: Load holiday data ===
        holidays_dim = load_nyc_holidays_dimension_recreated()
        
        # === STEP 3: Create temporal features for each month in 2023–2024 ===
        print(f"\n📊 Step 2: Creating temporal features for all months (2023-2024)...")
        
        years = [2023, 2024]
        months = list(range(1, 13))  # 1-12
        
        saved_files = []
        
        for year in years:
            for month in months:
                print(f"\n💾 Processing {year}-{month:02d}...")
                
                try:
                    # Create all dates for the month
                    from datetime import datetime, timedelta
                    import calendar
                    
                    # Get number of days in the month
                    days_in_month = calendar.monthrange(year, month)[1]
                    
                    # Create a DataFrame containing all timestamps in the month
                    dates = []
                    for day in range(1, days_in_month + 1):
                        for hour in range(24):
                            dates.append({
                                'pickup_datetime': datetime(year, month, day, hour, 0, 0),
                                'date_placeholder': 1  # placeholder to construct DataFrame
                            })
                    
                    # Convert to Spark DataFrame
                    schema = "pickup_datetime timestamp, date_placeholder int"
                    monthly_base = spark.createDataFrame(dates, schema=schema)
                    
                    # Add temporal features
                    monthly_temporal = add_temporal_features_recreated(monthly_base, holidays_dim)
                    
                    # Drop placeholder column; keep only temporal features
                    temporal_features_df = monthly_temporal.drop("date_placeholder")
                    
                    record_count = temporal_features_df.count()
                    
                    # Save path
                    save_path = f"./lake/external_features/temporal/year={year}/month={month:02d}"
                    
                    # Write as Parquet
                    temporal_features_df.write.mode("overwrite").parquet(save_path)
                    
                    saved_files.append({
                        'year': year,
                        'month': month,
                        'records': record_count,
                        'path': save_path
                    })
                    
                    print(f"✅ Saved {record_count:,} temporal records to: {save_path}")
                    
                except Exception as e:
                    print(f"❌ Failed to process {year}-{month:02d}: {e}")
        
        # === STEP 4: Create summary information ===
        print(f"\n📋 Step 3: Creating temporal features summary...")
        
        summary = {
            'total_months_saved': len(saved_files),
            'years_covered': years,
            'features_created': actual_temporal_features,
            'feature_count': len(actual_temporal_features),
            'monthly_files': saved_files,
            'feature_descriptions': {
                'pickup_date': 'Date extracted from pickup_datetime',
                'pickup_hour': 'Hour of day (0-23)',
                'pickup_day_of_week': 'Day of week (0=Sunday, 6=Saturday)',
                'pickup_month': 'Month number (1-12)',
                'pickup_year': 'Year (2023, 2024)',
                'is_weekend': 'Boolean flag for weekend (Saturday/Sunday)',
                'holiday_name': 'Name of federal holiday if applicable',
                'is_holiday': 'Boolean flag for federal holidays',
                'rush_hour_label': 'Detailed rush hour classification (morning_rush, evening_rush, business_hours, etc.)',
                'is_rush_hour': 'Boolean flag for peak hours (morning_rush + evening_rush)',
                'time_period': 'General time period (morning, afternoon, evening, night)'
            }
        }
        
        # Save summary information
        import json
        summary_path = "./lake/external_features/temporal/temporal_summary.json"
        import os
        os.makedirs(os.path.dirname(summary_path), exist_ok=True)
        
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"✅ Summary saved: {summary_path}")
        
        # === STEP 5: Display results ===
        print(f"\n🎉 TEMPORAL FEATURES PROCESSING COMPLETE!")
        print(f"="*60)
        print(f"📅 Years covered: {years}")
        print(f"📂 Months saved: {len(saved_files)}")
        print(f"🏷️ Features per month: {len(actual_temporal_features)}")
        
        print(f"\n📊 MONTHLY BREAKDOWN:")
        for file_info in saved_files[:6]:  # Show first 6 months
            print(f"  {file_info['year']}-{file_info['month']:02d}: {file_info['records']:,} records")
        if len(saved_files) > 6:
            print(f"  ... and {len(saved_files) - 6} more months")
        
        print(f"\n📅 TEMPORAL FEATURES CREATED:")
        for i, feature in enumerate(actual_temporal_features, 1):
            desc = summary['feature_descriptions'].get(feature, '')
            print(f"  {i:2d}. {feature:<20} - {desc}")
        
        return {
            'summary': summary,
            'saved_files': saved_files,
            'holidays_dim': holidays_dim
        }
        
    except Exception as e:
        print(f"❌ Temporal processing failed: {e}")
        import traceback
        traceback.print_exc()
        return None

def load_monthly_temporal_features(year, month):
    """
    加载指定年月的temporal特征数据
    
    Args:
        year (int): 年份
        month (int): 月份
    
    Returns:
        DataFrame: 该月的temporal特征数据
    """
    
    load_path = f"./lake/external_features/temporal/year={year}/month={month:02d}"
    
    try:
        temporal_data = spark.read.parquet(load_path)
        print(f"✅ Loaded temporal data for {year}-{month:02d}: {temporal_data.count():,} records")
        return temporal_data
    except Exception as e:
        print(f"❌ Failed to load temporal data for {year}-{month:02d}: {e}")
        return None

def test_temporal_data_files():
    """
    测试已保存的temporal数据文件
    """
    
    print("🔍 TESTING SAVED TEMPORAL DATA FILES")
    print("="*50)
    
    # 测试几个月份
    test_months = [(2023, 1), (2023, 6), (2024, 1), (2024, 12)]
    
    for year, month in test_months:
        try:
            temporal_data = load_monthly_temporal_features(year, month)
            if temporal_data:
                print(f"✅ {year}-{month:02d}: {temporal_data.count():,} records, {len(temporal_data.columns)} columns")
                
                # 显示sample
                if year == 2023 and month == 1:
                    print(f"\n📊 Sample data for January 2023:")
                    temporal_data.select([
                        "pickup_datetime", "pickup_hour", "is_weekend", 
                        "is_holiday", "rush_hour_label", "time_period"
                    ]).show(5, truncate=False)
        except Exception as e:
            print(f"❌ {year}-{month:02d}: {e}")

# ============================================================================
# 可执行代码
# ============================================================================

# 运行temporal数据处理和保存
print("🚀 STARTING TEMPORAL FEATURES PROCESSING AND MONTHLY STORAGE")
print("="*70)

temporal_result = create_monthly_temporal_features_and_save()

if temporal_result:
    print("\n🔍 Testing temporal data files...")
    test_temporal_data_files()
    
    print("\n🎉 ALL TEMPORAL PROCESSING COMPLETE!")
    print("📂 Temporal features are now available monthly for 2023-2024")
    print("🔗 Ready for integration with trip data")
else:
    print("\n❌ Temporal processing failed")

print("\n📁 USAGE:")
print("# To load specific month:")
print("# temporal_jan_2023 = load_monthly_temporal_features(2023, 1)")
print("# temporal_dec_2024 = load_monthly_temporal_features(2024, 12)")

In [ ]:
temporal_jan_2023 = load_monthly_temporal_features(2023, 1)
temporal_jan_2023.show()

In [ ]:
def visualize_weather_patterns_comprehensive(weather_df, save_plots=True):
    """
    Create comprehensive weather patterns visualization dashboard.
    
    Args:
        weather_df (DataFrame): Processed weather data
        save_plots (bool): Whether to save plots to file
    
    Returns:
        dict: Visualization results and insights
    """
    
    print("📊 CREATING COMPREHENSIVE WEATHER PATTERNS VISUALIZATION")
    print("="*60)
    
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        import pandas as pd
        import numpy as np
        from datetime import datetime
        
        # Enhanced styling
        plt.style.use('default')
        sns.set_palette("husl")
        plt.rcParams.update({
            'font.size': 10,
            'axes.titlesize': 12,
            'axes.labelsize': 11,
            'xtick.labelsize': 9,
            'ytick.labelsize': 9,
            'legend.fontsize': 9
        })
        
        # Convert to Pandas for visualization
        print(f"📈 Converting weather data for visualization...")
        viz_df = weather_df.toPandas()
        viz_df['weather_date'] = pd.to_datetime(viz_df['weather_date'])
        
        print(f"📊 Creating dashboard from {len(viz_df):,} daily weather records...")
        
        # Create comprehensive figure
        fig = plt.figure(figsize=(25, 20))
        
        # === 1. TEMPERATURE TRENDS OVER TIME ===
        plt.subplot(4, 4, 1)
        
        # Monthly temperature trends
        viz_df_monthly = viz_df.set_index('weather_date').resample('M').agg({
            'tmax_c': 'mean',
            'tmin_c': 'mean',
            'temp_avg_calculated': 'mean'
        })
        
        plt.plot(viz_df_monthly.index, viz_df_monthly['tmax_c'], 
                label='Max Temp', linewidth=2, color='red', alpha=0.8)
        plt.plot(viz_df_monthly.index, viz_df_monthly['tmin_c'], 
                label='Min Temp', linewidth=2, color='blue', alpha=0.8)
        plt.plot(viz_df_monthly.index, viz_df_monthly['temp_avg_calculated'], 
                label='Avg Temp', linewidth=2, color='green', alpha=0.8)
        
        plt.fill_between(viz_df_monthly.index, viz_df_monthly['tmin_c'], 
                        viz_df_monthly['tmax_c'], alpha=0.2, color='gray')
        
        plt.xlabel('Date')
        plt.ylabel('Temperature (°C)')
        plt.title('NYC Temperature Trends (2023-2024)\nMonthly Averages')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        
        # === 2. PRECIPITATION PATTERNS ===
        plt.subplot(4, 4, 2)
        
        # Monthly precipitation
        monthly_precip = viz_df.set_index('weather_date').resample('M')['precipitation_mm'].sum()
        
        bars = plt.bar(monthly_precip.index, monthly_precip.values, 
                      color='steelblue', alpha=0.7, edgecolor='navy')
        
        # Highlight heavy precipitation months
        for i, bar in enumerate(bars):
            if monthly_precip.values[i] > 150:  # >150mm per month
                bar.set_color('darkblue')
                bar.set_alpha(0.9)
        
        plt.xlabel('Date')
        plt.ylabel('Monthly Precipitation (mm)')
        plt.title('NYC Precipitation Patterns\n(Dark blue: Heavy months >150mm)')
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        # === 3. SEASONAL TEMPERATURE DISTRIBUTION ===
        plt.subplot(4, 4, 3)
        
        # Box plot by season
        season_order = ['winter', 'spring', 'summer', 'fall']
        season_data = [viz_df[viz_df['season'] == season]['tmax_c'].dropna() 
                      for season in season_order]
        
        box_plot = plt.boxplot(season_data, labels=season_order, patch_artist=True)
        
        # Color seasons
        colors = ['lightblue', 'lightgreen', 'orange', 'brown']
        for patch, color in zip(box_plot['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        plt.ylabel('Max Temperature (°C)')
        plt.title('Seasonal Temperature Distribution\n(Max Daily Temperature)')
        plt.grid(True, alpha=0.3)
        
        # === 4. WEATHER CONDITIONS FREQUENCY ===
        plt.subplot(4, 4, 4)
        
        # Weather conditions pie chart
        conditions = {
            'Clear/Mild': len(viz_df[(viz_df['severe_weather'] == 0) & 
                                   (viz_df['is_precipitation'] == 0)]),
            'Light Rain': viz_df['is_light_rain'].sum(),
            'Moderate Rain': viz_df['is_moderate_rain'].sum(),
            'Heavy Rain': viz_df['is_heavy_rain'].sum(),
            'Snow': viz_df['is_snow'].sum(),
            'Severe Weather': viz_df['severe_weather'].sum()
        }
        
        # Remove zero values
        conditions = {k: v for k, v in conditions.items() if v > 0}
        
        colors = ['gold', 'lightblue', 'blue', 'darkblue', 'white', 'red'][:len(conditions)]
        plt.pie(conditions.values(), labels=conditions.keys(), autopct='%1.1f%%', 
               colors=colors, startangle=90)
        plt.title('Weather Conditions Distribution\n(Daily Frequency)')
        
        # === 5. EXTREME WEATHER TIMELINE ===
        plt.subplot(4, 4, 5)
        
        # Mark extreme weather events
        extreme_dates = viz_df[viz_df['severe_weather'] == 1]['weather_date']
        extreme_temps = viz_df[viz_df['severe_weather'] == 1]['tmax_c']
        
        plt.scatter(viz_df['weather_date'], viz_df['tmax_c'], 
                   c='lightgray', alpha=0.5, s=20, label='Normal Days')
        
        if len(extreme_dates) > 0:
            plt.scatter(extreme_dates, extreme_temps, 
                       c='red', s=50, alpha=0.8, label='Severe Weather Days')
        
        plt.xlabel('Date')
        plt.ylabel('Max Temperature (°C)')
        plt.title('Extreme Weather Events Timeline\n(Red dots: Severe weather days)')
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        # === 6. WEATHER COMFORT INDEX DISTRIBUTION ===
        plt.subplot(4, 4, 6)
        
        comfort_dist = viz_df['weather_comfort_index'].value_counts().sort_index()
        comfort_labels = {1: "Very\nUncomfortable", 2: "Uncomfortable", 
                         3: "Moderate", 4: "Comfortable", 5: "Very\nComfortable"}
        
        bars = plt.bar(comfort_dist.index, comfort_dist.values, 
                      color=['red', 'orange', 'yellow', 'lightgreen', 'green'][:len(comfort_dist)],
                      alpha=0.7)
        
        plt.xlabel('Weather Comfort Index')
        plt.ylabel('Number of Days')
        plt.title('Weather Comfort Distribution\n(1=Very Uncomfortable, 5=Very Comfortable)')
        plt.xticks(comfort_dist.index, [comfort_labels.get(i, str(i)) for i in comfort_dist.index])
        plt.grid(True, alpha=0.3)
        
        # === 7. TEMPERATURE VS PRECIPITATION CORRELATION ===
        plt.subplot(4, 4, 7)
        
        # Scatter plot with color coding by season
        season_colors = {'winter': 'blue', 'spring': 'green', 'summer': 'red', 'fall': 'orange'}
        
        for season in viz_df['season'].unique():
            season_data = viz_df[viz_df['season'] == season]
            plt.scatter(season_data['tmax_c'], season_data['precipitation_mm'], 
                       c=season_colors.get(season, 'gray'), alpha=0.6, 
                       label=season.title(), s=30)
        
        plt.xlabel('Max Temperature (°C)')
        plt.ylabel('Daily Precipitation (mm)')
        plt.title('Temperature vs Precipitation\n(Colored by Season)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # === 8. DAILY TEMPERATURE RANGE ===
        plt.subplot(4, 4, 8)
        
        # Temperature range over time
        plt.plot(viz_df['weather_date'], viz_df['temp_range_c'], 
                color='purple', alpha=0.7, linewidth=1)
        
        # Add rolling average
        viz_df_sorted = viz_df.sort_values('weather_date')
        rolling_avg = viz_df_sorted['temp_range_c'].rolling(window=30, center=True).mean()
        plt.plot(viz_df_sorted['weather_date'], rolling_avg, 
                color='red', linewidth=2, label='30-day average')
        
        plt.xlabel('Date')
        plt.ylabel('Daily Temperature Range (°C)')
        plt.title('Daily Temperature Range\n(Max - Min Temperature)')
        plt.legend()
        plt.xticks(rotation=45)
        plt.grid(True, alpha=0.3)
        
        # === 9. MONTHLY WEATHER SUMMARY HEATMAP ===
        plt.subplot(4, 4, 9)
        
        # Create monthly summary
        viz_df['year_month'] = viz_df['weather_date'].dt.to_period('M')
        monthly_summary = viz_df.groupby('year_month').agg({
            'tmax_c': 'mean',
            'precipitation_mm': 'sum',
            'severe_weather': 'sum',
            'is_snow': 'sum'
        })
        
        # Create heatmap data
        heatmap_data = monthly_summary[['tmax_c']].T
        
        sns.heatmap(heatmap_data, cmap='RdYlBu_r', center=15, 
                   cbar_kws={'label': 'Avg Max Temp (°C)'})
        plt.xlabel('Month')
        plt.ylabel('')
        plt.title('Monthly Average Max Temperature\n(Heatmap)')
        plt.xticks(rotation=45)
        
        # === 10. WIND PATTERNS ===
        plt.subplot(4, 4, 10)
        
        if 'wind_speed_kmh' in viz_df.columns and viz_df['wind_speed_kmh'].notna().any():
            # Wind speed distribution
            wind_data = viz_df['wind_speed_kmh'].dropna()
            
            plt.hist(wind_data, bins=30, alpha=0.7, color='lightblue', edgecolor='blue')
            plt.axvline(wind_data.mean(), color='red', linestyle='--', 
                       label=f'Mean: {wind_data.mean():.1f} km/h')
            plt.axvline(20, color='orange', linestyle='--', 
                       label='Windy threshold: 20 km/h')
            
            plt.xlabel('Wind Speed (km/h)')
            plt.ylabel('Frequency')
            plt.title('Wind Speed Distribution')
            plt.legend()
            plt.grid(True, alpha=0.3)
        else:
            plt.text(0.5, 0.5, 'Wind Data\nNot Available', 
                    ha='center', va='center', transform=plt.gca().transAxes,
                    fontsize=14, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray"))
            plt.title('Wind Speed Distribution')
        
        # === 11. PRECIPITATION INTENSITY ANALYSIS ===
        plt.subplot(4, 4, 11)
        
        # Precipitation categories
        precip_categories = {
            'No Rain': (viz_df['precipitation_mm'] == 0).sum(),
            'Light Rain\n(0-2.5mm)': viz_df['is_light_rain'].sum(),
            'Moderate Rain\n(2.5-10mm)': viz_df['is_moderate_rain'].sum(), 
            'Heavy Rain\n(10-25mm)': viz_df['is_heavy_rain'].sum(),
            'Extreme Rain\n(>25mm)': viz_df['is_extreme_rain'].sum()
        }
        
        # Remove zero categories
        precip_categories = {k: v for k, v in precip_categories.items() if v > 0}
        
        bars = plt.bar(range(len(precip_categories)), precip_categories.values(), 
                      color=['gold', 'lightblue', 'blue', 'darkblue', 'purple'][:len(precip_categories)],
                      alpha=0.7)
        
        plt.xlabel('Precipitation Category')
        plt.ylabel('Number of Days')
        plt.title('Precipitation Intensity Distribution')
        plt.xticks(range(len(precip_categories)), precip_categories.keys(), rotation=45)
        plt.grid(True, alpha=0.3)
        
        # === 12. SEASONAL PRECIPITATION PATTERNS ===
        plt.subplot(4, 4, 12)
        
        seasonal_precip = viz_df.groupby('season')['precipitation_mm'].agg(['sum', 'mean'])
        
        x_pos = range(len(seasonal_precip))
        bars = plt.bar(x_pos, seasonal_precip['sum'], 
                      color=['lightblue', 'lightgreen', 'orange', 'brown'], alpha=0.7)
        
        # Add average as text on bars
        for i, (bar, avg_val) in enumerate(zip(bars, seasonal_precip['mean'])):
            plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                    f'Avg: {avg_val:.1f}mm', ha='center', va='bottom', fontsize=8)
        
        plt.xlabel('Season')
        plt.ylabel('Total Precipitation (mm)')
        plt.title('Seasonal Precipitation Totals\n(With daily averages)')
        plt.xticks(x_pos, seasonal_precip.index)
        plt.grid(True, alpha=0.3)
        
        # === 13. TEMPERATURE EXTREMES BY MONTH ===
        plt.subplot(4, 4, 13)
        
        monthly_extremes = viz_df.groupby(viz_df['weather_date'].dt.month).agg({
            'tmax_c': ['min', 'max'],
            'tmin_c': ['min', 'max']
        })
        
        months = range(1, 13)
        month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                      'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        
        # Plot temperature ranges
        plt.fill_between(months, 
                        monthly_extremes[('tmin_c', 'min')],
                        monthly_extremes[('tmax_c', 'max')],
                        alpha=0.3, color='gray', label='Temperature Range')
        
        plt.plot(months, monthly_extremes[('tmax_c', 'max')], 
                'r-o', label='Highest Max', linewidth=2)
        plt.plot(months, monthly_extremes[('tmin_c', 'min')], 
                'b-o', label='Lowest Min', linewidth=2)
        
        plt.xlabel('Month')
        plt.ylabel('Temperature (°C)')
        plt.title('Monthly Temperature Extremes\n(Highest/Lowest recorded)')
        plt.xticks(months, month_names)
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # === 14. WEATHER IMPACT SEVERITY ===
        plt.subplot(4, 4, 14)
        
        # Weather impact categories for ride-hailing
        impact_data = {
            'Severe Weather': viz_df['severe_weather'].sum(),
            'Heavy Precipitation': (viz_df['is_heavy_rain'] | viz_df['is_extreme_rain']).sum(),
            'Snow Days': viz_df['is_snow'].sum(),
            'Extreme Cold': viz_df['is_very_cold'].sum(),
            'Extreme Heat': viz_df['is_extreme_heat'].sum(),
            'High Winds': viz_df['high_winds'].sum() if 'high_winds' in viz_df.columns else 0,
            'Fog/Low Visibility': viz_df['fog_mist'].sum() if 'fog_mist' in viz_df.columns else 0
        }
        
        # Remove zero values
        impact_data = {k: v for k, v in impact_data.items() if v > 0}
        
        if impact_data:
            plt.barh(range(len(impact_data)), list(impact_data.values()), 
                    color='red', alpha=0.7)
            plt.yticks(range(len(impact_data)), list(impact_data.keys()))
            plt.xlabel('Number of Days')
            plt.title('Weather Impact on Transportation\n(Potential high-demand conditions)')
            plt.grid(True, alpha=0.3)
        else:
            plt.text(0.5, 0.5, 'No Severe\nWeather Detected', 
                    ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('Weather Impact Analysis')
        
        # === 15. COMFORT INDEX TRENDS ===
        plt.subplot(4, 4, 15)
        
        # Monthly comfort index trends
        monthly_comfort = viz_df.set_index('weather_date').resample('M')['weather_comfort_index'].mean()
        
        plt.plot(monthly_comfort.index, monthly_comfort.values, 
                'o-', linewidth=2, markersize=6, color='green')
        
        # Add comfort zones
        plt.axhline(y=3, color='orange', linestyle='--', alpha=0.7, label='Moderate')
        plt.axhline(y=4, color='lightgreen', linestyle='--', alpha=0.7, label='Comfortable')
        
        plt.xlabel('Date')
        plt.ylabel('Average Comfort Index')
        plt.title('Monthly Weather Comfort Trends\n(1=Very Uncomfortable, 5=Very Comfortable)')
        plt.xticks(rotation=45)
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.ylim(1, 5)
        
        # === 16. SUMMARY STATISTICS ===
        plt.subplot(4, 4, 16)
        
        # Calculate key statistics
        stats_text = f"""NYC WEATHER SUMMARY (2023-2024)
        
📊 DATASET: {len(viz_df):,} days
📅 PERIOD: {viz_df['weather_date'].min().strftime('%Y-%m-%d')} to {viz_df['weather_date'].max().strftime('%Y-%m-%d')}

🌡️ TEMPERATURE:
Max: {viz_df['tmax_c'].max():.1f}°C ({viz_df['tmax_c'].max()*9/5+32:.1f}°F)
Min: {viz_df['tmin_c'].min():.1f}°C ({viz_df['tmin_c'].min()*9/5+32:.1f}°F)
Avg: {viz_df['tmax_c'].mean():.1f}°C ({viz_df['tmax_c'].mean()*9/5+32:.1f}°F)

🌧️ PRECIPITATION:
Total: {viz_df['precipitation_mm'].sum():.0f}mm
Rainy days: {viz_df['is_precipitation'].sum()} ({viz_df['is_precipitation'].mean()*100:.1f}%)
Heavy rain days: {viz_df['is_heavy_rain'].sum()} ({viz_df['is_heavy_rain'].mean()*100:.1f}%)

❄️ SNOW:
Snowy days: {viz_df['is_snow'].sum()} ({viz_df['is_snow'].mean()*100:.1f}%)

⚠️ SEVERE WEATHER:
Severe weather days: {viz_df['severe_weather'].sum()} ({viz_df['severe_weather'].mean()*100:.1f}%)

😊 COMFORT:
Avg comfort index: {viz_df['weather_comfort_index'].mean():.1f}/5
Very comfortable days: {(viz_df['weather_comfort_index']==5).sum()} ({(viz_df['weather_comfort_index']==5).mean()*100:.1f}%)
        """
        
        plt.text(0.05, 0.95, stats_text, transform=plt.gca().transAxes, 
                fontsize=9, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgray", alpha=0.8))
        plt.axis('off')
        plt.title('Summary Statistics', pad=20, fontweight='bold')
        
        # Enhanced layout
        plt.tight_layout(pad=4.0, h_pad=4.0, w_pad=3.0)
        
        # Save the plot
        if save_plots:
            output_file = "weather_patterns_comprehensive_dashboard.png"
            plt.savefig(output_file, dpi=300, bbox_inches='tight', 
                       facecolor='white', edgecolor='none')
            print(f"💾 Saved comprehensive weather dashboard: {output_file}")
        
        plt.show()
        
        # === GENERATE BUSINESS INSIGHTS ===
        insights = generate_weather_business_insights(viz_df)
        
        return {
            'total_days': len(viz_df),
            'date_range': (viz_df['weather_date'].min(), viz_df['weather_date'].max()),
            'temperature_stats': {
                'max_temp': viz_df['tmax_c'].max(),
                'min_temp': viz_df['tmin_c'].min(),
                'avg_temp': viz_df['tmax_c'].mean()
            },
            'precipitation_stats': {
                'total_precipitation': viz_df['precipitation_mm'].sum(),
                'rainy_days': viz_df['is_precipitation'].sum(),
                'heavy_rain_days': viz_df['is_heavy_rain'].sum()
            },
            'severe_weather_days': viz_df['severe_weather'].sum(),
            'avg_comfort_index': viz_df['weather_comfort_index'].mean(),
            'business_insights': insights
        }
        
    except Exception as e:
        print(f"❌ Weather visualization failed: {e}")
        import traceback
        print(f"Traceback: {traceback.format_exc()}")
        return None

def generate_weather_business_insights(weather_df):
    """
    Generate business insights for ride-hailing operations.
    """
    
    insights = []
    
    try:
        # Peak demand weather conditions
        severe_weather_pct = weather_df['severe_weather'].mean() * 100
        if severe_weather_pct > 5:
            insights.append(f"High severe weather frequency ({severe_weather_pct:.1f}%) - significant surge pricing opportunities")
        
        # Rain impact
        rainy_days_pct = weather_df['is_precipitation'].mean() * 100
        insights.append(f"Rain occurs {rainy_days_pct:.1f}% of days - consistent demand boost factor")
        
        # Seasonal patterns
        summer_comfort = weather_df[weather_df['season'] == 'summer']['weather_comfort_index'].mean()
        winter_comfort = weather_df[weather_df['season'] == 'winter']['weather_comfort_index'].mean()
        
        if summer_comfort > winter_comfort + 0.5:
            insights.append("Summer significantly more comfortable than winter - expect seasonal demand variations")
        
        # Extreme weather days
        extreme_days = (weather_df['is_extreme_heat'].sum() + 
                       weather_df['is_very_cold'].sum() + 
                       weather_df['is_heavy_rain'].sum() + 
                       weather_df['is_snow'].sum())
        extreme_pct = extreme_days / len(weather_df) * 100
        
        insights.append(f"Extreme weather occurs {extreme_pct:.1f}% of days - plan driver incentives accordingly")
        
        # Comfort optimization
        very_comfortable_pct = (weather_df['weather_comfort_index'] == 5).mean() * 100
        insights.append(f"Perfect weather days: {very_comfortable_pct:.1f}% - potential for outdoor activities partnerships")
        
    except Exception as e:
        insights.append(f"Insight generation error: {str(e)}")
    
    return insights

def test_weather_features_with_trip_data(year=2023, month=1, sample_fraction=0.005):
    """
    Test weather features integration with actual trip data.
    """
    
    print("🌤️ TESTING WEATHER FEATURES WITH TRIP DATA")
    print("="*50)
    
    # === STEP 1: LOAD PROCESSED WEATHER DATA ===
    print("🌤️ Step 1: Using processed weather data...")
    weather_data = processed_weather  # From previous processing
    
    # === STEP 2: LOAD SAMPLE TRIP DATA ===
    print(f"\n📂 Step 2: Loading sample trip data...")
    trip_data = extract_uber_data_for_analysis(
        year_list=[year],
        month_list=[month],
        sample_fraction=sample_fraction
    )
    
    if trip_data is None:
        print("❌ Failed to load trip data")
        return None
    
    # === STEP 3: ADD WEATHER FEATURES TO TRIPS ===
    print(f"\n🌤️ Step 3: Adding weather features to trips...")
    
    # Extract pickup date for weather join
    trip_with_weather = (trip_data
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
        .join(broadcast(weather_data), 
              F.col("pickup_date") == F.col("weather_date"), "left")
        .drop("weather_date")
    )
    
    # === STEP 4: ANALYZE WEATHER-TRIP PATTERNS ===
    print(f"\n📈 Step 4: Analyzing weather-trip relationships...")
    
    sample_df = trip_with_weather.limit(10000).toPandas()
    
    print(f"📊 Weather-enhanced trip sample: {len(sample_df):,} trips")
    
    # Weather coverage
    weather_coverage = sample_df['tmax_c'].notna().sum() / len(sample_df) * 100
    print(f"🌤️ Weather data coverage: {weather_coverage:.1f}%")
    
    # Sample analysis
    if weather_coverage > 50:
        print(f"\n🔍 WEATHER-TRIP INSIGHTS:")
        
        # Average trip characteristics by weather
        weather_groups = sample_df.groupby('weather_comfort_index').agg({
            'trip_miles': 'mean',
            'trip_time': 'mean'
        }).round(2)
        
        print(f"Trip patterns by weather comfort:")
        print(weather_groups)
        
        # Rainy day analysis
        if 'is_precipitation' in sample_df.columns:
            rainy_trips = sample_df[sample_df['is_precipitation'] == 1]
            clear_trips = sample_df[sample_df['is_precipitation'] == 0]
            
            if len(rainy_trips) > 10 and len(clear_trips) > 10:
                print(f"\n🌧️ RAINY DAY ANALYSIS:")
                print(f"Avg trip miles - Rainy: {rainy_trips['trip_miles'].mean():.2f}, Clear: {clear_trips['trip_miles'].mean():.2f}")
                print(f"Avg trip time - Rainy: {rainy_trips['trip_time'].mean():.0f}s, Clear: {clear_trips['trip_time'].mean():.0f}s")
    
    # === STEP 5: SHOW SAMPLE DATA ===
    print(f"\n📋 Step 5: Sample of weather-enhanced trips...")
    
    weather_trip_cols = ['pickup_datetime', 'trip_miles', 'trip_time', 
                        'tmax_c', 'precipitation_mm', 'is_precipitation', 
                        'severe_weather', 'weather_comfort_index']
    
    try:
        sample_display = trip_with_weather.select(weather_trip_cols).limit(5).toPandas()
        print(sample_display.to_string(index=False))
    except Exception as e:
        print(f"Could not display sample: {e}")
    
    print(f"\n✅ WEATHER-TRIP INTEGRATION COMPLETE!")
    
    return trip_with_weather

# ============================================================================
# RUN COMPREHENSIVE WEATHER ANALYSIS
# ============================================================================

print("📊 RUNNING COMPREHENSIVE WEATHER ANALYSIS...")
print("Creating dashboard, insights, and testing trip integration")
print("-" * 70)

# Create comprehensive visualization
weather_viz_results = visualize_weather_patterns_comprehensive(
    processed_weather, 
    save_plots=True
)

if weather_viz_results:
    print(f"\n🎉 WEATHER VISUALIZATION SUCCESS!")
    print(f"📊 Dashboard created from {weather_viz_results['total_days']:,} days of data")
    print(f"📅 Period: {weather_viz_results['date_range'][0]} to {weather_viz_results['date_range'][1]}")
    
    print(f"\n💡 KEY BUSINESS INSIGHTS:")
    for insight in weather_viz_results['business_insights'][:4]:
        print(f"  🎯 {insight}")
    
    # Test integration with trip data
    print(f"\n🔄 Testing weather integration with trip data...")
    weather_trip_result = test_weather_features_with_trip_data(
        year=2023, month=1, sample_fraction=0.005
    )
    
    if weather_trip_result:
        print(f"✅ Weather-trip integration successful!")
        print(f"🎯 Ready for profitability analysis with weather factors")
    
else:
    print("❌ Weather visualization failed")

print(f"\n📁 FILES CREATED:")
print(f"  📊 weather_patterns_comprehensive_dashboard.png - 16-panel weather analysis")
print(f"  🌤️ Complete weather feature set ready for dispatch optimization")

In [ ]:
def create_monthly_weather_features_and_save():
    """
    Create monthly weather features based on the NOAA section and save outputs.
    Reproduce the weather feature set defined in External Data: NOAA.
    """
    
    print("🌤️ CREATING AND SAVING MONTHLY WEATHER FEATURES")
    print("Based on actual NOAA processing code from External Data section")
    print("="*70)
    
    # === Weather features to create (from code) ===
    actual_weather_features = [
        # Temperature features
        'tmax_c', 'tmin_c', 'temp_avg_calculated', 'temp_range_c',
        
        # Precipitation features
        'precipitation_mm', 'is_precipitation', 'is_light_rain', 
        'is_moderate_rain', 'is_heavy_rain', 'is_extreme_rain',
        
        # Temperature categories
        'is_very_cold', 'is_cold', 'is_cool', 'is_mild', 'is_warm', 'is_hot', 'is_very_hot',
        
        # Weather conditions
        'severe_weather', 'weather_comfort_index', 'fog_mist', 'thunder',
        
        # Wind and others
        'wind_speed_ms', 'is_windy',
        
        # Season
        'season'
    ]
    
    print(f"📋 Will create {len(actual_weather_features)} weather features")
    
    try:
        # === STEP 1: 运行原始的天气处理函数 ===
        print("\n🔧 Step 1: Running original weather processing...")
        processed_weather = process_noaa_weather_data_optimized()
        
        if processed_weather is None:
            print("❌ Failed to process weather data")
            return None
            
        total_days = processed_weather.count()
        print(f"✅ Processed weather data: {total_days} days")
        
        # === STEP 2: 为每个月创建数据 ===
        print("\n📅 Step 2: Creating monthly weather datasets...")
        
        # 创建年月列
        monthly_weather = (processed_weather
            .withColumn("year", F.year("weather_date"))
            .withColumn("month", F.month("weather_date"))
        )
        
        # 获取所有年月组合
        year_months = (monthly_weather
            .select("year", "month")
            .distinct()
            .orderBy("year", "month")
            .collect()
        )
        
        print(f"📊 Found data for {len(year_months)} year-month combinations")
        
        # === STEP 3: 按月保存 ===
        saved_files = []
        
        for row in year_months:
            year = row['year']
            month = row['month']
            
            print(f"\n💾 Processing {year}-{month:02d}...")
            
            # 过滤当月数据
            monthly_data = monthly_weather.filter(
                (F.col("year") == year) & (F.col("month") == month)
            )
            
            monthly_count = monthly_data.count()
            
            # 保存路径
            save_path = f"./lake/external_features/weather/year={year}/month={month:02d}"
            
            try:
                # 保存为parquet
                monthly_data.write.mode("overwrite").parquet(save_path)
                
                saved_files.append({
                    'year': year,
                    'month': month, 
                    'days': monthly_count,
                    'path': save_path
                })
                
                print(f"✅ Saved {monthly_count} days to: {save_path}")
                
            except Exception as e:
                print(f"❌ Failed to save {year}-{month:02d}: {e}")
        
        # === STEP 4: 创建汇总信息 ===
        print(f"\n📋 Step 4: Creating summary...")
        
        summary = {
            'total_months_saved': len(saved_files),
            'date_range': {
                'start': f"{min(year_months, key=lambda x: (x['year'], x['month']))['year']}-{min(year_months, key=lambda x: (x['year'], x['month']))['month']:02d}",
                'end': f"{max(year_months, key=lambda x: (x['year'], x['month']))['year']}-{max(year_months, key=lambda x: (x['year'], x['month']))['month']:02d}"
            },
            'features_created': actual_weather_features,
            'feature_count': len(actual_weather_features),
            'monthly_files': saved_files
        }
        
        # 保存汇总信息
        import json
        summary_path = "./lake/external_features/weather/weather_summary.json"
        import os
        os.makedirs(os.path.dirname(summary_path), exist_ok=True)
        
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=2)
        
        print(f"✅ Summary saved: {summary_path}")
        
        # === STEP 5: 显示结果 ===
        print(f"\n🎉 WEATHER DATA PROCESSING COMPLETE!")
        print(f"="*50)
        print(f"📅 Period: {summary['date_range']['start']} to {summary['date_range']['end']}")
        print(f"📂 Months saved: {summary['total_months_saved']}")
        print(f"🏷️ Features per month: {summary['feature_count']}")
        
        print(f"\n📊 MONTHLY BREAKDOWN:")
        for file_info in saved_files:
            print(f"  {file_info['year']}-{file_info['month']:02d}: {file_info['days']} days")
        
        print(f"\n🌤️ WEATHER FEATURES CREATED:")
        for i, feature in enumerate(actual_weather_features, 1):
            print(f"  {i:2d}. {feature}")
        
        return {
            'processed_weather': processed_weather,
            'summary': summary,
            'saved_files': saved_files
        }
        
    except Exception as e:
        print(f"❌ Weather processing failed: {e}")
        import traceback
        traceback.print_exc()
        return None

def load_monthly_weather_features(year, month):
    """
    加载指定年月的天气特征数据
    
    Args:
        year (int): 年份
        month (int): 月份
    
    Returns:
        DataFrame: 该月的天气特征数据
    """
    
    load_path = f"./lake/external_features/weather/year={year}/month={month:02d}"
    
    try:
        weather_data = spark.read.parquet(load_path)
        print(f"✅ Loaded weather data for {year}-{month:02d}: {weather_data.count()} days")
        return weather_data
    except Exception as e:
        print(f"❌ Failed to load weather data for {year}-{month:02d}: {e}")
        return None

def verify_weather_features_consistency():
    """
    验证所有月份的天气特征一致性
    """
    
    print("🔍 VERIFYING WEATHER FEATURES CONSISTENCY")
    print("="*50)
    
    # 读取汇总信息
    try:
        import json
        with open("./lake/external_features/weather/weather_summary.json", 'r') as f:
            summary = json.load(f)
    except Exception as e:
        print(f"❌ Could not read summary: {e}")
        return False
    
    expected_features = set(summary['features_created'])
    inconsistent_months = []
    
    for file_info in summary['monthly_files']:
        year = file_info['year']
        month = file_info['month']
        
        try:
            monthly_data = load_monthly_weather_features(year, month)
            if monthly_data is not None:
                actual_features = set(monthly_data.columns)
                
                # 检查特征一致性
                missing_features = expected_features - actual_features
                extra_features = actual_features - expected_features
                
                if missing_features or extra_features:
                    inconsistent_months.append({
                        'year_month': f"{year}-{month:02d}",
                        'missing': list(missing_features),
                        'extra': list(extra_features)
                    })
        except Exception as e:
            print(f"⚠️ Could not verify {year}-{month:02d}: {e}")
    
    if inconsistent_months:
        print(f"❌ Found {len(inconsistent_months)} months with inconsistent features:")
        for month_info in inconsistent_months:
            print(f"  {month_info['year_month']}: missing={month_info['missing']}, extra={month_info['extra']}")
        return False
    else:
        print(f"✅ All {len(summary['monthly_files'])} months have consistent features")
        return True

# ============================================================================
# 可执行代码
# ============================================================================

# 运行天气数据处理和保存
print("🚀 STARTING WEATHER DATA PROCESSING AND MONTHLY STORAGE")
print("="*70)

weather_result = create_monthly_weather_features_and_save()

if weather_result:
    print("\n🔍 Verifying feature consistency across months...")
    consistency_check = verify_weather_features_consistency()
    
    if consistency_check:
        print("\n🎉 ALL WEATHER PROCESSING COMPLETE AND VERIFIED!")
        print("📂 Weather features are now available monthly from 2023-2024")
        print("🔗 Ready for integration with trip data")
    else:
        print("\n⚠️ Feature consistency issues detected - may need investigation")
else:
    print("\n❌ Weather processing failed")

print("\n📁 USAGE:")
print("# To load specific month:")
print("# weather_jan_2023 = load_monthly_weather_features(2023, 1)")
print("# weather_dec_2024 = load_monthly_weather_features(2024, 12)")

In [ ]:
def test_weather_data_files():
    """
    Simple test function: open and inspect saved monthly weather data files
    """
    
    print("🔍 TESTING SAVED WEATHER DATA FILES")
    print("="*50)
    
    # === 测试1: 检查文件是否存在 ===
    print("📂 Step 1: Checking file existence...")
    
    test_months = [
        (2023, 1),   # 2023年1月
        (2023, 6),   # 2023年6月  
        (2024, 1),   # 2024年1月
        (2024, 12)   # 2024年12月
    ]
    
    for year, month in test_months:
        file_path = f"./lake/external_features/weather/year={year}/month={month:02d}"
        
        try:
            # 尝试读取文件
            weather_data = spark.read.parquet(file_path)
            day_count = weather_data.count()
            col_count = len(weather_data.columns)
            
            print(f"✅ {year}-{month:02d}: {day_count} days, {col_count} columns")
            
        except Exception as e:
            print(f"❌ {year}-{month:02d}: File not found or error - {e}")
    
    # === 测试2: 详细查看一个月的数据 ===
    print(f"\n📊 Step 2: Detailed view of January 2023...")
    
    try:
        jan_2023 = spark.read.parquet("./lake/external_features/weather/year=2023/month=01")
        
        print(f"📋 January 2023 weather data:")
        print(f"   Records: {jan_2023.count()}")
        print(f"   Columns: {len(jan_2023.columns)}")
        
        # 显示列名
        print(f"\n🏷️ Column names:")
        for i, col in enumerate(jan_2023.columns, 1):
            print(f"   {i:2d}. {col}")
        
        # 显示样本数据
        print(f"\n📖 Sample data (first 5 rows):")
        jan_2023.select([
            "weather_date", "tmax_c", "tmin_c", "precipitation_mm", 
            "is_precipitation", "severe_weather", "weather_comfort_index", "season"
        ]).show(5, truncate=False)
        
        # 显示统计信息
        print(f"\n📈 Quick statistics:")
        stats_df = jan_2023.select([
            "tmax_c", "tmin_c", "precipitation_mm", "weather_comfort_index"
        ]).toPandas()
        
        print(stats_df.describe().round(2))
        
    except Exception as e:
        print(f"❌ Could not read January 2023 data: {e}")
    
    # === 测试3: 检查汇总文件 ===
    print(f"\n📋 Step 3: Checking summary file...")
    
    try:
        import json
        with open("./lake/external_features/weather/weather_summary.json", 'r') as f:
            summary = json.load(f)
        
        print(f"✅ Summary file found:")
        print(f"   Total months: {summary['total_months_saved']}")
        print(f"   Date range: {summary['date_range']['start']} to {summary['date_range']['end']}")
        print(f"   Feature count: {summary['feature_count']}")
        
        print(f"\n🌤️ Weather features available:")
        for i, feature in enumerate(summary['features_created'][:10], 1):
            print(f"   {i:2d}. {feature}")
        if len(summary['features_created']) > 10:
            print(f"   ... and {len(summary['features_created']) - 10} more")
        
    except Exception as e:
        print(f"❌ Could not read summary file: {e}")
    
    print(f"\n✅ WEATHER DATA TEST COMPLETE!")

# 运行测试
test_weather_data_files()

### ⚠️ IMPORTANT: Pipeline Updated

The pipeline has been rewritten for correctness, robustness, and reproducibility.

- **Key fixes**
  - **Correct revenue**: Uber commission = `base_passenger_fare − driver_pay`
  - **Early filtering**: Remove negative `uber_commission`, then winsorize via `approxQuantile` (default 1%/99%; thresholds logged; tunable)
  - **Robust readers**: Recursive lookup + `pathGlobFilter="*.parquet"` (ignores Windows `:Zone.Identifier` artifacts)
  - **Temporal join fixed**: Join on `pickup_date + pickup_hour` when available, else dedupe to one row per date
  - **Spatial/Weather joins**: Spatial on `PULocationID/DOLocationID`; weather uses optimized dataset with date join and coverage checks
  - **Feature cleanup**: Drop pass-through fees and old incorrect revenue metrics; remove redundant/correlated features during optimization
  - **Reproducible from raw**: Starts at `./lake/raw/hvfhs/v1`; no intermediate dependencies
  - **Transparent logging**: Clear “REMOVAL SUMMARY” (before/after counts, negatives removed) and quantile thresholds

- **Usage (single month)**
```python
import importlib
import preprocessing.final_corrected_profitability_analysis.data_pipeline as dp
dp = importlib.reload(dp)

df = dp.load_and_integrate_all_features_final(
    2023, 10,
    sample_fraction=1.0,              # 1.0 for full run
    diagnostics=True,
    enable_commission_filter=True,
    low_quantile=0.01, high_quantile=0.99,
    enable_winsorization=True,
)
dp.save_comprehensive_profitability_dataset_final(df, 2023, 10, dataset_name="corrected_profitability_2023_10")
```

- **Usage (batch)**
```python
results = dp.process_all_months_profitability_batch(
    years=[2023, 2024],
    months=list(range(1, 13))
)
```

- **Outputs**
  - Saved to `./lake/comprehensive/<dataset_name>/year=YYYY/month=MM/`
  - Negatives after cleanup: `0`
  - Removal summary printed at end (meets reporting spec)

- **Backups**
  - Old notebook helpers: `old_pipeline_functions_from_notebook.py`
  - (Code history retains earlier pipeline versions)

- **Quick validation**
```python
# After pipeline, should be 0
df.filter("uber_commission < 0").count()
```

- **Notes**
  - Readers use `recursiveFileLookup` + `pathGlobFilter="*.parquet"` throughout
  - Quantile clipping may log zero clipped rows when min == q_low (often 0 after removing negatives) or when q_high == max due to ties

- **Example results (2023‑10)**
  - Raw: 20,186,330 → Uber-only: 14,375,602 → Quality: 14,374,798  
  - Commission cleanup: removed 1,543,943 negatives → Final: 12,830,855 (≈49–50 features after optimization)

### quick smoke test using small sample of the data


In [ ]:
# 1) 导入与Spark会话（确保你已先运行高内存Spark初始化单元）
from preprocessing.final_corrected_profitability_analysis.data_pipeline import (
    get_spark_session,
    load_and_integrate_all_features_final,
    save_comprehensive_profitability_dataset_final,
)

import importlib
import preprocessing.final_corrected_profitability_analysis.data_pipeline as dp
dp = importlib.reload(dp)

spark = get_spark_session()
spark.sparkContext.setLogLevel("WARN")  # 可选：降低日志噪音

# 2) 参数区
YEAR = 2023
MONTH = 1
SAMPLE_FRACTION = 0.01     # 小样本冒烟: 0.01；全量: 1.0
LOW_Q, HIGH_Q = 0.01, 0.99 # 分位数截尾阈值
ENABLE_FILTER = True       # 是否启用基于 uber_commission 的“负值剔除+截尾”

# 3) 跑数据流水线（包含：JOIN前早过滤 + 外部特征整合 + 去重优化）
df = load_and_integrate_all_features_final(
    YEAR, MONTH,
    sample_fraction=SAMPLE_FRACTION,
    diagnostics=True,
    enable_commission_filter=ENABLE_FILTER,
    low_quantile=LOW_Q,
    high_quantile=HIGH_Q,
    enable_winsorization=True,
)

print("rows =", df.count(), "cols =", len(df.columns))

# 4) 可选：保存（小样本/全量，按需改名）
DATASET_NAME = "optimized_profitability_sample" if SAMPLE_FRACTION < 1.0 else f"corrected_profitability_{YEAR}_{MONTH:02d}"
save_path = save_comprehensive_profitability_dataset_final(df, YEAR, MONTH, dataset_name=DATASET_NAME)
print("saved_to:", save_path)

In [ ]:
df.printSchema()
df.show(5)


In [ ]:
from pyspark.sql import functions as F

spatial_path = "./lake/external_features/spatial/year=2023/month=10"
s = spark.read.option("recursiveFileLookup","true").option("pathGlobFilter","*.parquet").parquet(spatial_path)

# Inspect columns
print(s.columns)

# Check duplicates on the PU/DO pair
s.groupBy("PULocationID","DOLocationID").count().orderBy(F.desc("count")).show(10)
s.groupBy("PULocationID","DOLocationID").count().where("count > 1").count()

In [7]:
# Running for one month
import importlib
import preprocessing.final_corrected_profitability_analysis.data_pipeline as dp
dp = importlib.reload(dp)

df = dp.load_and_integrate_all_features_final(
    2023, 4, sample_fraction=1.0, diagnostics=True,
    enable_commission_filter=True, low_quantile=0.01, high_quantile=0.99,
    enable_winsorization=True, write_reports=True,  # 打开自动写报表
)

🔄 COMPLETE REWRITTEN PIPELINE FOR 2023-04
📂 Loading raw HVFHS data for 2023-04...
   Raw HVFHS records: 19,144,903
🚕 Filtering for Uber trips...
   Uber trips filtered: 13,998,413
🔍 Applying data quality filters...


   After quality filters: 13,986,649
📊 Using full dataset (no sampling)
🗑️ Dropping unwanted features...
   Dropped core unwanted: ['hvfhs_license_num', 'dispatching_base_num', 'originating_base_num', 'request_datetime']
   Dropped pass-through fees: ['tolls', 'bcf', 'sales_tax', 'congestion_surcharge', 'airport_fee', 'tips']
🔧 Adding CORRECTED computed columns...
📊 Sample revenue statistics:


   Average passenger base fare: $25.04
   Average driver pay: $19.97
   Average Uber commission: $5.07
   Average Uber take rate: 19.0%
✅ CORRECTED Base Uber data loaded successfully: 13,986,649 trips, 23 columns
✅ Base data loaded: 13,986,649 trips, 23 columns
🧹 Removing negative uber_commission rows: 1,796,399


✂️ Winsorizing uber_commission to [0.0000, 446.1700] (below: 0, above: 0)
✅ Spatial features available: ./lake/external_features/spatial/year=2023/month=04
✅ Temporal features available: ./lake/external_features/temporal/year=2023/month=04
✅ Weather features available: ./lake/external_features/weather/year=2023/month=04
✅ Spatial features integrated via PU/DO LocationID
  ℹ️ Temporal daily dedup applied: 689 extra rows collapsed
✅ Temporal features integrated via ['pickup_date']
🌤️ Integrating weather features...
  ✅ Using optimized weather dataset (base): ./lake/external_features/weather_optimized
  📊 Weather data (full): 731 records, 10 columns
  📆 Filtered weather to 2023-04: 30 records
  📋 Weather columns: ['weather_date', 'season', 'temp_avg_calculated', 'is_freezing', 'is_hot', 'precipitation_mm', 'is_snow', 'wind_speed_kmh', 'weather_comfort_index', 'fog_flag']
  📅 Weather date columns found: ['weather_date']
  🔑 Using weather_date for join
  🔍 Checking date overlap...
    Base 

  ✅ time_period provides additional information - keeping
✅ Feature optimization complete:
  📊 Final features: 46
  🗑️ Features dropped: 8
  📈 Reduction: 14.8%
📊 Features after optimization: 46
📈 Optimization applied: 8 features removed

🧾 REMOVAL SUMMARY (this run)
----------------------------------------------------------------------
Raw records: 19,144,903
After provider filter (Uber only): 13,998,413  | Removed: 5,146,490
After quality filters: 13,986,649         | Removed: 11,764
After commission cleanup: 12,190,250        | Removed: 1,796,399 (negatives: 1,796,399)

🎯 COMPLETE PIPELINE FINISHED!
  📊 Final dataset: 12,190,250 trips, 46 total columns
  🔗 Data integrity: 87.2% trips retained
  💰 Revenue calculation: CORRECTED (Uber commission-based)
  🔑 Key Uber revenue columns: ['uber_commission', 'uber_take_rate', 'uber_revenue_per_mile', 'uber_revenue_per_minute']
  🗑️ Features cleaned: core unwanted, pass-through fees, old revenue metrics
  ✅ Spatial join: Uses PULocationID/DOLo

📝 Removal summary written: ./lake/reports/removal_summary/year=2023/month=04


ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/home/yunka/project-1-individual-samhouhaoyang/.venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/yunka/project-1-individual-samhouhaoyang/.venv/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [11]:
df.show(10)
df.printSchema()

25/08/28 17:07:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------------+------------+-------------------+-------------------+-------------------+----------+---------+-------------------+----------+-----------------+---------------------+------------------+------------------+------------------+---------------------+-----------------------+------------------+----------+--------------------+---------------+----------+-------------------+---------------+-------------+----------------+------------------+----------------+-----------+------------------+------------+-----------+----------+------------+----------+---------------+-----------+------+-------------------+-----------+------+----------------+-------+------------------+---------------------+--------+
|pickup_date|PULocationID|DOLocationID|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|trip_miles|trip_time|base_passenger_fare|driver_pay|shared_match_flag|trip_duration_minutes|     avg_speed_mph|   uber_commission|    uber_take_rate|uber_revenue_per_mile|uber_revenue_p

In [ ]:
# read the report
import glob, pandas as pd
csv_path = glob.glob("./lake/reports/removal_summary/year=2023/month=01_csv/*.csv")[0]
pd.read_csv(csv_path)

,raw_count,uber_count,quality_count,sampled_count,removed_non_uber,removed_quality,removed_sampling,removed_neg_commission,winsor_low,winsor_high,year,month,final_rows
0,18479031,13580152,13566695,13566695,4898879,13457,0,1457498,0.0,935.86,2023,1,12109197


### Full pipeline for all months with the new data pipeline

In [ ]:
# Run full pipeline for 2023–2024 and write monthly reports
import importlib
import preprocessing.final_corrected_profitability_analysis.data_pipeline as dp
dp = importlib.reload(dp)

spark = dp.get_spark_session()
spark.sparkContext.setLogLevel("WARN")

years = [2023,2024]
months = list(range(1, 12+1))

# 批量处理与保存（每月自动写入 removal/coverage 报表）
results = dp.process_all_months_profitability_batch(years=years, months=months)

# 汇总运行结果
ok = [k for k,v in results.items() if v.get('status') == 'success']
fail = {k:v.get('error') for k,v in results.items() if v.get('status') == 'failed'}
total = sum(v.get('records', 0) for v in results.values() if v.get('status') == 'success')
print("success_months:", len(ok))
print("total_records:", total)
if fail:
    print("failed_months:", fail)

# 合并并查看两年删除/过滤指标（Parquet）
removals = (
    spark.read
         .option("recursiveFileLookup","true")
         .option("pathGlobFilter","*.parquet")
         .parquet("./lake/reports/removal_summary")
)
removals.orderBy("year","month").show(50, truncate=False)
removals.coalesce(1).write.mode("overwrite").option("header", True).csv("./lake/reports/removal_summary_2023_2024_csv")

# 合并并查看两年覆盖率（可选）
coverage = (
    spark.read
         .option("recursiveFileLookup","true")
         .option("pathGlobFilter","*.parquet")
         .parquet("./lake/reports/coverage")
)
coverage.orderBy("year","month","group","column").show(50, truncate=False)
coverage.coalesce(1).write.mode("overwrite").option("header", True).csv("./lake/reports/coverage_2023_2024_csv")

### Year-level preprocessing removals & retention summary

This section aggregates monthly removal reports into year-level totals and computes retention/removal percentages per step. It:
- Reads `lake/reports/removal_summary` (Parquet) for completed months
- Sums counts by `year` and computes step-wise retention/removal rates
- Adds an “ALL” row (both years combined)
- Optionally writes a CSV to `lake/reports/removal_summary_by_year`

#### Columns (counts)
- raw: loaded rows from HVFHS (all providers)
- uber: after provider filter (Uber only, HV0003)
- quality: after quality filters (required fields, valid ranges, time order)
- sampled: after sampling (if no sampling, equals quality)
- final: final rows after downstream cleanup (e.g., negative commission removals; joins don’t change row count)
- rm_non_uber / rm_quality / rm_sampling / rm_neg_comm: rows removed at each step

#### Columns (percentages)
- ret_after_uber_pct: uber ÷ raw (retention after provider filter)
- ret_after_quality_pct: quality ÷ uber
- ret_after_sampling_pct: sampled ÷ quality
- ret_final_pct: final ÷ sampled
- ret_final_vs_raw_pct: final ÷ raw (overall retention from start to finish)
- pct_rm_non_uber: rm_non_uber ÷ raw
- pct_rm_quality: rm_quality ÷ uber
- pct_rm_sampling: rm_sampling ÷ quality
- pct_rm_neg_comm: rm_neg_comm ÷ sampled

#### How to interpret
- Percentages are per-step with that step’s input as denominator; don’t sum percentages across steps.
- ret_final_vs_raw_pct shows overall retained share from raw to final.
- Values are rounded to 0.1%, so tiny removals may appear as 100.0% retention.
- Only months that successfully wrote reports are included (missing months are excluded).

In [6]:
from pyspark.sql import functions as F

rem = (spark.read
       .option("recursiveFileLookup","true")
       .option("pathGlobFilter","*.parquet")
       .parquet("./lake/reports/removal_summary"))

base = (rem.groupBy("year").agg(
    F.sum("raw_count").alias("raw"),
    F.sum("uber_count").alias("uber"),
    F.sum("quality_count").alias("quality"),
    F.sum("sampled_count").alias("sampled"),
    F.sum("final_rows").alias("final"),
    F.sum("removed_non_uber").alias("rm_non_uber"),
    F.sum("removed_quality").alias("rm_quality"),
    F.sum("removed_sampling").alias("rm_sampling"),
    F.sum("removed_neg_commission").alias("rm_neg_comm"),
))

def pct(num_col, den_col):
    return F.round(100 * F.when(F.col(den_col) > 0, F.col(num_col) / F.col(den_col)).otherwise(None), 1)

res = (base
    .withColumn("ret_after_uber_pct", pct("uber","raw"))
    .withColumn("ret_after_quality_pct", pct("quality","uber"))
    .withColumn("ret_after_sampling_pct", pct("sampled","quality"))
    .withColumn("ret_final_pct", pct("final","sampled"))
    .withColumn("ret_final_vs_raw_pct", pct("final","raw"))
    .withColumn("pct_rm_non_uber", pct("rm_non_uber","raw"))
    .withColumn("pct_rm_quality", pct("rm_quality","uber"))
    .withColumn("pct_rm_sampling", pct("rm_sampling","quality"))
    .withColumn("pct_rm_neg_comm", pct("rm_neg_comm","sampled"))
)

tot = (res.agg(
    F.sum("raw").alias("raw"),
    F.sum("uber").alias("uber"),
    F.sum("quality").alias("quality"),
    F.sum("sampled").alias("sampled"),
    F.sum("final").alias("final"),
    F.sum("rm_non_uber").alias("rm_non_uber"),
    F.sum("rm_quality").alias("rm_quality"),
    F.sum("rm_sampling").alias("rm_sampling"),
    F.sum("rm_neg_comm").alias("rm_neg_comm"),
).withColumn("year", F.lit("ALL"))
 .withColumn("ret_after_uber_pct", pct("uber","raw"))
 .withColumn("ret_after_quality_pct", pct("quality","uber"))
 .withColumn("ret_after_sampling_pct", pct("sampled","quality"))
 .withColumn("ret_final_pct", pct("final","sampled"))
 .withColumn("ret_final_vs_raw_pct", pct("final","raw"))
 .withColumn("pct_rm_non_uber", pct("rm_non_uber","raw"))
 .withColumn("pct_rm_quality", pct("rm_quality","uber"))
 .withColumn("pct_rm_sampling", pct("rm_sampling","quality"))
 .withColumn("pct_rm_neg_comm", pct("rm_neg_comm","sampled"))
)

cols = ["year","raw","uber","quality","sampled","final",
        "rm_non_uber","rm_quality","rm_sampling","rm_neg_comm",
        "ret_after_uber_pct","ret_after_quality_pct","ret_after_sampling_pct",
        "ret_final_pct","ret_final_vs_raw_pct",
        "pct_rm_non_uber","pct_rm_quality","pct_rm_sampling","pct_rm_neg_comm"]

# Cast year to string before union to avoid NumberFormatException
res_str = res.withColumn("year", F.col("year").cast("string"))

final = res_str.select(cols).unionByName(tot.select(cols))
final.orderBy(F.when(F.col("year")=="ALL", 1).otherwise(0), F.col("year")).show(truncate=False)

# Optional: save
final.coalesce(1).write.mode("overwrite").option("header", True).csv("./lake/reports/removal_summary_by_year")

+----+---------+---------+---------+---------+---------+-----------+----------+-----------+-----------+------------------+---------------------+----------------------+-------------+--------------------+---------------+--------------+---------------+---------------+
|year|raw      |uber     |quality  |sampled  |final    |rm_non_uber|rm_quality|rm_sampling|rm_neg_comm|ret_after_uber_pct|ret_after_quality_pct|ret_after_sampling_pct|ret_final_pct|ret_final_vs_raw_pct|pct_rm_non_uber|pct_rm_quality|pct_rm_sampling|pct_rm_neg_comm|
+----+---------+---------+---------+---------+---------+-----------+----------+-----------+-----------+------------------+---------------------+----------------------+-------------+--------------------+---------------+--------------+---------------+---------------+
|2023|211973723|152853704|152778818|152778818|135187730|59120019   |74886     |0          |17591088   |72.1              |100.0                |100.0                 |88.5         |63.8                |

In [ ]:
y, m = 2024, 9  # check the data
path = f"./lake/comprehensive/corrected_profitability_{y}_{m:02d}"
df = spark.read.option("recursiveFileLookup","true").parquet(path)
df.printSchema()
print("cols =", len(df.columns))
print(df.columns)  #

root
 |-- pickup_date: date (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: integer (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)
 |-- uber_commission: double (nullable = true)
 |-- uber_take_rate: double (nullable = true)
 |-- uber_revenue_per_mile: double (nullable = true)
 |-- uber_revenue_per_minute: double (nullable = true)
 |-- driver_efficiency: double (nullable = true)
 |-- PU_Borough: string (nullable = true)
 |-- PU_Zone: string (nullable = true)
 |-- PU_service_zone: string (nullable = tru